In this notebook we are going to compute different connectivity measures and aggregate them into a dataframe. 

In [ ]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

In [ ]:
import numpy as np

import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns
from scipy.stats import ttest_rel
import pandas as pd
import os 
from tqdm import tqdm
from IPython.display import clear_output
import warnings 

# Add in my own local version of mne-connectivity
import sys 
sys.path.insert(0, '/Users/salmanqasim/Documents/GitRepos/mne-connectivity')
from mne_connectivity.effective import phase_slope_index_time

from mne_connectivity import seed_target_indices

# I only want to see warnings once
warnings.filterwarnings('ignore')


Note: If you have installed this package in editable form on Minerva, you must append the local path! This is because Minerva requires that you point your package installs away from the local directory for space reasons, but editable packages have to be installed locally.

In [ ]:
import sys
sys.path.append('/Users/salmanqasim/Documents/GitRepos/LFPAnalysis')

In [ ]:
from LFPAnalysis import oscillation_utils, statistics_utils


In [ ]:
import mne

## First, get the behavior data

This is from another notebook with a kernel and environment specifically for beahvior

In [ ]:
base_dir = '/Volumes/T7 Shield' # this is the root directory for most un-archived data and results
load_dir = f'{base_dir}/work/qasims01/MemoryBanditData/EMU'

# Load the demographics data to get information about who has run the task
demo_file = f'{base_dir}/projects/guLab/Salman/EMU/MemoryBanditsDatabase.xlsx'
demo_data = pd.read_excel(demo_file, engine='openpyxl')
subj_df = demo_data[(demo_data.LongVersion==1) & (demo_data.Minerva==1)]

# Let's get rid of Iowa patients for now 

# subj_df = subj_df[subj_df.Site=='UI']
subj_df = subj_df[~subj_df['MSSMCode'].isin(['MS015', 'MS024', 'MS034', 'MS038'])]


days = ['Day1', 'Day2']
subj_ids = subj_df.MSSMCode.values
subj_formats = subj_df.format.values
subj_sites = subj_df.Site.values
subj_days = subj_df.Day2.values

In [ ]:
subj_ids

In [ ]:
# load behavioral data
learn_df = pd.read_csv(f'{load_dir}/learn_df_RWH.csv')
combined_df = pd.read_csv(f'{load_dir}/full_df_RWH.csv')

learn_df['gamble_rt_log'] = np.log(learn_df.gamble_rt)

combined_df['gamble_rt_log'] = np.log(combined_df.gamble_rt_gamble)
combined_df['recog_rt_log'] = np.log(combined_df.recog_rt)

combined_df['memory_outcome'] = combined_df[['hits', 'misses', 'correct_rejections', 
                                       'false_alarms']].idxmax(1).to_frame('memory_outcome') 

In [ ]:
# First, let's drop reaction times that are physiologically too fast
learn_df = learn_df[learn_df.gamble_rt>0.3]
drop_trials = pd.concat([combined_df[combined_df.gamble_rt_gamble<0.3], 
                          combined_df[combined_df.recog_rt<0.3]])
combined_df.drop(index=drop_trials.index, inplace=True)

# Next, let's drop reaction times that are slower than the 97.5th percentile of each subject's reaction times
for subj in learn_df.participant.unique():
    rt_threshold = np.percentile(learn_df[learn_df.participant==subj].gamble_rt, 95)
    learn_df.loc[(learn_df.participant==subj) & (learn_df.gamble_rt>rt_threshold), 'gamble_rt'] = np.nan
    combined_df.loc[(combined_df.participant==subj) & (combined_df.gamble_rt_gamble>rt_threshold), 'gamble_rt_gamble'] = np.nan
    learn_df.loc[(learn_df.participant==subj), 'rpe_cat'] = pd.qcut(learn_df.loc[(learn_df.participant==subj), 'rpe'], 3, labels=['neg', 'neutral', 'pos'])
    combined_df.loc[(combined_df.participant==subj), 'rpe_cat'] = pd.qcut(combined_df.loc[(combined_df.participant==subj), 'rpe'], 3, labels=['neg', 'neutral', 'pos'])

learn_df.dropna(subset=['gamble_rt'], inplace=True)
combined_df.dropna(subset=['gamble_rt_gamble'], inplace=True)

# 3/18/2025

# recover the latent Q-values
combined_df['Q'] = combined_df.rpe + combined_df.reward_gamble
# compute the prior trial's rpe:
combined_df['prior_rpe'] = combined_df.groupby('participant')['rpe'].shift(1)
# compute the trials since last reversal
reversal_trials = [16, 33, 48, 65]
combined_df['trials_since_reversal'] = combined_df.apply(lambda x: np.mod(x.trials_gamble, reversal_trials[int(x.block_gamble-2)]) if x.block_gamble>1 else x.trials_gamble, axis=1)
# recode hits as 0.5 and misses as -0.5
combined_df['phit'] = combined_df.apply(lambda x: 0.5 if x.hits==True else -0.5 if x.hits==False else np.nan, axis=1)
# recode reward  as 0.5 and non-reward as -0.5
combined_df['preward'] = combined_df.apply(lambda x: 0.5 if x.reward_gamble==1 else -0.5 if x.reward_gamble==0 else np.nan, axis=1)
# recode choice==1 as 0.5 and choice==0 as -0.5
combined_df['pgamble'] = combined_df.apply(lambda x: 0.5 if x.gamble_gamble==1 else -0.5 if x.gamble_gamble==0 else np.nan, axis=1)

# Gelman standardization 
for subj in combined_df.participant.unique():

        combined_df.loc[combined_df.participant==subj, 
                'zQ'] = (combined_df.loc[combined_df.participant==subj, 
                                                                        'Q'] - np.nanmean(combined_df.loc[combined_df.participant==subj, 
                                                                                                'Q'])) / (2*np.nanstd(combined_df.loc[combined_df.participant==subj, 
                                                                                                                                'Q']))
        combined_df.loc[combined_df.participant==subj,
                'zprior_rpe'] = (combined_df.loc[combined_df.participant==subj, 
                                                                        'prior_rpe'] - np.nanmean(combined_df.loc[combined_df.participant==subj, 
                                                                                                'prior_rpe'])) / (2*np.nanstd(combined_df.loc[combined_df.participant==subj, 
                                                                                                                                'prior_rpe']))
        combined_df.loc[combined_df.participant==subj, 
                        'zrpe'] = (combined_df.loc[combined_df.participant==subj, 
                                                                                'rpe'] - np.nanmean(combined_df.loc[combined_df.participant==subj, 
                                                                                                        'rpe'])) / (2*np.nanstd(combined_df.loc[combined_df.participant==subj, 
                                                                                                                                        'rpe']))
        combined_df.loc[combined_df.participant==subj, 
                'zpm'] = (combined_df.loc[combined_df.participant==subj, 
                                                                        'DPRIME'] - np.nanmean(combined_df.loc[combined_df.participant==subj, 
                                                                                                'DPRIME'])) / (2*np.nanstd(combined_df.loc[combined_df.participant==subj, 
                                                                                                                                'DPRIME']))

        combined_df.loc[combined_df.participant==subj,
                'ztrials_since_reversal'] = (combined_df.loc[combined_df.participant==subj, 
                                                                        'trials_since_reversal'] - np.nanmean(combined_df.loc[combined_df.participant==subj, 
                                                                                                'trials_since_reversal'])) / (2*np.nanstd(combined_df.loc[combined_df.participant==subj, 
                                                                                                                                'trials_since_reversal']))
        # compute the reaction-time based EV with res[ect to each subject's min and max RT
        RT_inferred_EV_subj =  1 - ((combined_df.loc[combined_df.participant==subj, 
                                                     'gamble_rt_log'] - np.nanmin(combined_df.loc[combined_df.participant==subj, 
                                                                                                  'gamble_rt_log'])) / (np.nanmax(combined_df.loc[combined_df.participant==subj, 
                                                                                                                                                  'gamble_rt_log']) - np.nanmin(combined_df.loc[combined_df.participant==subj, 
                                                                                                                                                                                                'gamble_rt_log'])))
        # compute 
        combined_df.loc[combined_df.participant==subj,
                'RT_rpe'] = (combined_df.loc[combined_df.participant==subj, 
                                                                        'reward_gamble'] - RT_inferred_EV_subj)

        combined_df.loc[combined_df.participant==subj,
                'zRT_rpe'] = (combined_df.loc[combined_df.participant==subj, 
                                                                        'RT_rpe'] - np.nanmean(combined_df.loc[combined_df.participant==subj, 
                                                                                                'RT_rpe'])) / (2*np.nanstd(combined_df.loc[combined_df.participant==subj, 
                                                                                                                                'RT_rpe']))


In [ ]:
# 3/18/2025

# recover the latent Q-values
learn_df['Q'] = learn_df.rpe + learn_df.reward
# compute the prior trial's rpe:
learn_df['prior_rpe'] = learn_df.groupby('participant')['rpe'].shift(1)
# compute the trials since last reversal
reversal_trials = [16, 33, 48, 65]
learn_df['trials_since_reversal'] = learn_df.apply(lambda x: np.mod(x.trials, reversal_trials[int(x.block-2)]) if x.block>1 else x.trials, axis=1)
# recode reward  as 0.5 and non-reward as -0.5
learn_df['preward'] = learn_df.apply(lambda x: 0.5 if x.reward==1 else -0.5 if x.reward==0 else np.nan, axis=1)
# recode choice==1 as 0.5 and choice==0 as -0.5
learn_df['pgamble'] = learn_df.apply(lambda x: 0.5 if x.gamble==1 else -0.5 if x.gamble==0 else np.nan, axis=1)

# Gelman standardization 
for subj in learn_df.participant.unique():

        learn_df.loc[learn_df.participant==subj, 
                'zQ'] = (learn_df.loc[learn_df.participant==subj, 
                                                                        'Q'] - np.nanmean(learn_df.loc[learn_df.participant==subj, 
                                                                                                'Q'])) / (2*np.nanstd(learn_df.loc[learn_df.participant==subj, 
                                                                                                                                'Q']))
        learn_df.loc[learn_df.participant==subj,
                'zprior_rpe'] = (learn_df.loc[learn_df.participant==subj, 
                                                                        'prior_rpe'] - np.nanmean(learn_df.loc[learn_df.participant==subj, 
                                                                                                'prior_rpe'])) / (2*np.nanstd(learn_df.loc[learn_df.participant==subj, 
                                                                                                                                'prior_rpe']))
        learn_df.loc[learn_df.participant==subj, 
                        'zrpe'] = (learn_df.loc[learn_df.participant==subj, 
                                                                                'rpe'] - np.nanmean(learn_df.loc[learn_df.participant==subj, 
                                                                                                        'rpe'])) / (2*np.nanstd(learn_df.loc[learn_df.participant==subj, 
                                                                                                                                        'rpe']))
        learn_df.loc[learn_df.participant==subj, 
                'zpm'] = (learn_df.loc[learn_df.participant==subj, 
                                                                        'DPRIME'] - np.nanmean(learn_df.loc[learn_df.participant==subj, 
                                                                                                'DPRIME'])) / (2*np.nanstd(learn_df.loc[learn_df.participant==subj, 
                                                                                                                                'DPRIME']))

        learn_df.loc[learn_df.participant==subj,
                'ztrials_since_reversal'] = (learn_df.loc[learn_df.participant==subj, 
                                                                        'trials_since_reversal'] - np.nanmean(learn_df.loc[learn_df.participant==subj, 
                                                                                                'trials_since_reversal'])) / (2*np.nanstd(learn_df.loc[learn_df.participant==subj, 
                                                                                                                                'trials_since_reversal']))
        # compute the reaction-time based EV with res[ect to each subject's min and max RT
        RT_inferred_EV_subj =  1 - ((learn_df.loc[learn_df.participant==subj, 
                                                     'gamble_rt_log'] - np.nanmin(learn_df.loc[learn_df.participant==subj, 
                                                                                                  'gamble_rt_log'])) / (np.nanmax(learn_df.loc[learn_df.participant==subj, 
                                                                                                                                                  'gamble_rt_log']) - np.nanmin(learn_df.loc[learn_df.participant==subj, 
                                                                                                                                                                                                'gamble_rt_log'])))
        # compute 
        learn_df.loc[learn_df.participant==subj,
                'RT_rpe'] = (learn_df.loc[learn_df.participant==subj, 
                                                                        'reward'] - RT_inferred_EV_subj)

        learn_df.loc[learn_df.participant==subj,
                'zRT_rpe'] = (learn_df.loc[learn_df.participant==subj, 
                                                                        'RT_rpe'] - np.nanmean(learn_df.loc[learn_df.participant==subj, 
                                                                                                'RT_rpe'])) / (2*np.nanstd(learn_df.loc[learn_df.participant==subj, 
                                                                                                                                'RT_rpe']))




In [ ]:
# Define parameters for the nested loop
event = 'feedback_start'
target_region = 'HPC'  # Fixed target region
source_regions = ['ACC', 'OFC', 'dmPFC', 'dlPFC']  # Source regions to loop over

# Initialize list to store all results
te_results = []

# Loop over subjects
for subj_id in tqdm(subj_ids, desc='Subjects'): 
    
    # Try to load TFR data for this subject
    filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/TFR'
    tfr_data = mne.time_frequency.read_tfrs(f'{filepath}/{event}-tfr.h5')
    
    # Get electrode df for this subject
    loadpath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}'
    elec_df = pd.read_csv(f'{loadpath}/Day1_reref_elec_df')
    
    # Get trial indices for each RPE category
    subj_learn = learn_df[learn_df.participant == subj_id]
        
    pos_rpe_index = subj_learn[subj_learn.rpe_cat == 'pos'].trials.values.astype(int) - 1
    neg_rpe_index = subj_learn[subj_learn.rpe_cat == 'neg'].trials.values.astype(int) - 1
    neutral_rpe_index = subj_learn[subj_learn.rpe_cat == 'neutral'].trials.values.astype(int) - 1
    
    # Loop over source regions
    for source_region in source_regions:
        
        # Get seed-target indices for this region pair
        try:
            seed_target_df = oscillation_utils.make_seed_target_df(
                elec_df, tfr_data, source_region, target_region
            )
        except Exception as e:
            print(f"Could not create seed-target df for {subj_id}, {source_region}->{target_region}: {e}")
            continue
        
        # Loop over hemispheres
        for hemi in ['l', 'r']:
            # Check if this hemisphere has valid electrodes
            if hemi not in seed_target_df.index.tolist():
                continue
            if len(seed_target_df['seed'][hemi]) == 0 or len(seed_target_df['target'][hemi]) == 0:
                continue
                
            seed_to_target = seed_target_indices(
                seed_target_df['seed'][hemi],
                seed_target_df['target'][hemi]
            )
            
            # Compute TE for each RPE category
            rpe_categories = {
                'pos': pos_rpe_index,
                'neg': neg_rpe_index,
                'neutral': neutral_rpe_index
            }
            
            for rpe_cat, trial_indices in rpe_categories.items():
                # Skip if no trials for this category
                if len(trial_indices) == 0:
                    continue
                freq_tes, freqs = oscillation_utils.compute_te(
                    tfr_data.data[trial_indices, :, :, :],
                    indices=seed_to_target,
                    buf_ms=0,
                    te_k=1,
                    surr_method='swap_time_blocks',
                    parallelize=True,
                    n_surr=5,
                    return_freqs=True,
                    net=True,
                    sfreq=tfr_data.info['sfreq'],
                    freqs=tfr_data.freqs
                )
                
                # Average across pairs if multiple pairs exist
                te_mean = np.nanmean(freq_tes, axis=0) if freq_tes.ndim > 1 else freq_tes
                
                # Store result for each frequency
                for fi, freq in enumerate(freqs):
                    te_results.append({
                        'subject': subj_id,
                        'source_region': source_region,
                        'target_region': target_region,
                        'hemisphere': hemi,
                        'rpe_category': rpe_cat,
                        'frequency': freq,
                        'te_zscore': te_mean[fi],
                        'n_pairs': freq_tes.shape[0] if freq_tes.ndim > 1 else 1,
                        'n_trials': len(trial_indices)
                    })
                        
    clear_output(wait=True)

# Convert to DataFrame
te_df = pd.DataFrame(te_results)
print(f"Collected {len(te_df)} observations across {te_df.subject.nunique()} subjects")


In [ ]:
te_results

In [ ]:
plt.plot(te_mean)

In [ ]:
sns.lineplot(data=te_df[te_df.source_region=='ACC'], 
x='frequency', y='te_zscore', hue='rpe_category', errorbar=None)



In [ ]:
sns.barplot(data=te_df[te_df.source_region=='ACC'], 
x='freq_band', y='te_zscore', hue='rpe_category', errorbar='se', hue_order=['neg', 'neutral', 'pos'])

# Encoding connectivity:

In [ ]:
# Explicitly define a list of analysis events and the baseline event. Should correspond to the dict

evs = {'gamble_start':[-1.0, 0], 
       'feedback_start': [0, 1.5],
       'baseline_start': [0, 0.75],
       'cue_on': [0, 1]}

baseline_ev = 'baseline_start'
analysis_evs = ['feedback_start']

day = 'Day1'

for i in analysis_evs: 
    if i not in evs.keys(): 
        raise ValueError('This analysis event is not in the dictionary')

## Experimental: Directed Connectivity (PSI) - computed over timepoints

3/24/2025: This runs either very slowly, or it takes a lot of compute to run in parallel. Let's try to speed this up/ make it more efficient. 

Also, I need to make sure I only remove the trials from after I compute it. 

In [ ]:
def run_surr_psi_time(mne_data, indices, band, freqs, n_cycles, buf_ms, n_surr, surr_method):

    n_pairs = len(indices)

    pairwise_connectivity = np.squeeze(phase_slope_index_time(mne_data,
                        indices=indices,
                        sfreq=mne_data.info['sfreq'],
                        mode="cwt_morlet",
                        fmin=band[0],
                        fmax=band[1],
                        mt_bandwidth=None,
                        freqs=freqs,
                        n_cycles=n_cycles,
                        padding=(buf_ms / 1000),
                        n_jobs=1,
                        verbose=None).get_data())

    if n_pairs == 1:
        # reshape data
        pairwise_connectivity = pairwise_connectivity.reshape((pairwise_connectivity.shape[0], n_pairs))

    if n_surr > 0:
        surr_mne = oscillation_utils.make_surrogate_data(mne_data,
            method=surr_method, n_shuffles=n_surr, return_generator=False)

        surr_struct = np.zeros([pairwise_connectivity.shape[0], pairwise_connectivity.shape[1], n_surr]) # allocate space for all the surrogates 

        for ns in range(n_surr): 

            surr_conn = np.squeeze(phase_slope_index_time(surr_mne[ns],
                        indices=indices,
                        sfreq=mne_data.info['sfreq'],
                        mode="cwt_morlet",
                        fmin=band[0],
                        fmax=band[1],
                        mt_bandwidth=None,
                        freqs=freqs,
                        n_cycles=n_cycles,
                        padding=(buf_ms / 1000),
                        n_jobs=1,
                        verbose=None).get_data())
            
            if n_pairs == 1:
                surr_conn = surr_conn.reshape((surr_conn.shape[0], n_pairs))
            
            surr_struct[:, :, ns] = surr_conn
                        
        surr_mean = np.nanmean(surr_struct, axis=-1)
        surr_std = np.nanstd(surr_struct, axis=-1)
        pairwise_connectivity = (pairwise_connectivity - surr_mean) / (surr_std)
    
    return pairwise_connectivity

In [ ]:
analysis_evs = ['cue_on']

In [ ]:
metric = 'psi_time'
freqs = np.logspace(*np.log10([2, 200]), num=30)
n_cycles = np.floor(np.logspace(*np.log10([3, 10]), num=30))

freq_dict = {'theta':[4, 8],
             'alpha':[8, 12],
             'beta':[12, 30],
             'gamma':[30, 70],
             'HFA':[70, 200]}

band1 = 'theta'
band2 = 'theta'

buf_ms = 1000
n_surr = 0

day = 'Day1'

savepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/Connectivity/scratch/{metric}'
if not os.path.exists(savepath):
    os.makedirs(savepath)

2/26/2025: I am specifically finding that the first half of the encoding period is when memory and reward are both eliciting coherence between HPC and PFC. So, I am going to restrict my analysis to the first half by trimming 750 ms off the back-end of the epochs (but keeping the buffer the same)

In [ ]:
from joblib import Parallel, delayed

def process_subject(subj_id, source_region, target_region, analysis_evs, 
                    base_dir, day, freq_dict, band1, freqs, n_cycles, buf_ms, n_surr, subj_df, learn_df, combined_df, metric):
    conn_group_data = []
    for event in analysis_evs:
        loadpath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}'
        epochs_reref = mne.read_epochs(f'{loadpath}/neural/{day}/{event}-epo.fif', preload=True) 

         # Let's get rid of the trials we have pre-emptively dropped 
        all_trials = np.arange(epochs_reref._data.shape[0])
        valid_trials = learn_df[learn_df.participant==subj_id].trials.values
        valid_trials_index = valid_trials -1 
        bad_trial_index = list( set(valid_trials_index) ^ set(all_trials))
        epochs_reref.drop(bad_trial_index, reason='Bad Behavior')

        # Trim the epochs by 750 ms
        # epochs_reref = epochs_reref.crop(tmin=-1, tmax=1.75)

        # Get electrode df 
        elec_df = pd.read_csv(f'{loadpath}/Day1_reref_elec_df')
        seed_target_df = oscillation_utils.make_seed_target_df(elec_df, epochs_reref, source_region, target_region)
        
        # For within-region connectivity, let's drop elements that are the same
        if len(seed_target_df) == 0:
            print('only within channel metrics')
            continue
            
        for hemi in ['l', 'r']:
            if hemi not in seed_target_df.index.tolist():
                continue
            else:
                seed_to_target = seed_target_indices(
                    seed_target_df['seed'][hemi],
                    seed_target_df['target'][hemi])
            
            pwise = run_surr_psi_time(epochs_reref.copy(),
                                      indices=seed_to_target,
                                      band=freq_dict[band1],
                                      freqs=freqs[(freqs >= freq_dict[band1][0]) & (freqs <= freq_dict[band1][1])],
                                      n_cycles=n_cycles[(freqs >= freq_dict[band1][0]) & (freqs <= freq_dict[band1][1])],
                                      buf_ms=buf_ms,
                                      n_surr=n_surr,
                                      surr_method='swap_time_blocks')
            print(pwise)
            
            pwise_dfs = []

            npairs = len(seed_to_target[0])
                
            for pair in range(npairs): 
                if epochs_reref.ch_names[seed_to_target[0][pair]] == epochs_reref.ch_names[seed_to_target[1][pair]]:
                    print('failed channel match')
                    continue
                pwise_df = pd.DataFrame(columns=['participant', 'roi1', 'roi2', 'hemi', 'pair_label', 'metric', 'event', 'conn'])
                pwise_df['conn'] = pwise[:, pair] 
                pwise_df['participant'] = subj_id
                pwise_df['age'] = subj_df[subj_df.MSSMCode==subj_id].Age.values[0]
                pwise_df['sex'] = subj_df[subj_df.MSSMCode==subj_id].Sex.str.strip().values[0]
                pwise_df['roi1'] = source_region
                pwise_df['roi2'] = target_region
                pwise_df['hemi'] = hemi 
                pwise_df['metric'] = metric
                pwise_df['event'] = event
                pwise_df['band0'] = band1
                pwise_df['band1'] =  band2
                
                pwise_df['pair_label'] = f'{epochs_reref.ch_names[seed_to_target[0][pair]]}:{epochs_reref.ch_names[seed_to_target[1][pair]]}'

                if event in ['gamble_start', 'feedback_start', 'baseline_start']:
                    pwise_df['trials'] = valid_trials
                    pwise_df = pwise_df.merge(learn_df[(learn_df.participant==subj_id)], on=['participant', 'trials'])
                elif event in ['cue_on', 'recog_time']:
                    combined_df = combined_df[combined_df.recog_rt>1] 
                    pwise_df['trials_mem'] =  np.arange(1, pwise_df.shape[0]+1)
                    # combined_df[(combined_df.participant==subj_id) & (combined_df.condition=='Day1')].sort_values(by='trials_mem').reset_index(drop=True).trials_mem.values
                    pwise_df = pwise_df.merge(combined_df[(combined_df.participant==subj_id) & (combined_df.condition=='Day1')].sort_values(by='trials_mem').reset_index(drop=True), 
                                                on=['participant', 'trials_mem'])


                pwise_dfs.append(pwise_df)
            pwise_dfs = pd.concat(pwise_dfs)
            conn_group_data.append(pwise_dfs)
    return conn_group_data

In [ ]:
# test_results = process_subject('MS035', '', target_region, analysis_evs, 
#                     base_dir, day, freq_dict, band1, freqs, n_cycles, buf_ms, n_surr, subj_df, learn_df, combined_df, metric)

In [ ]:
%%capture

pairs =(['ACC', 'HPC'], )

# pairs =(['HPC', 'OFC'], ['HPC', 'ACC'], ['HPC', 'dmPFC'], ['HPC', 'dlPFC'])
#  (['HPC', 'ACC'], ['HPC', 'dmPFC'], ['HPC', 'dlPFC'], ['HPC', 'OFC'])
# ['HPC', 'ACC'], ['HPC', 'dmPFC'], ['HPC', 'dlPFC']

for pair in pairs:
    source_region = pair[0]
    target_region = pair[1]
    
    results = Parallel(n_jobs=3)(delayed(process_subject)(
        subj_id, source_region, target_region, analysis_evs, base_dir, day, freq_dict, band1, freqs, n_cycles, buf_ms, n_surr, subj_df, learn_df, combined_df,  
        metric) for subj_id in tqdm(subj_ids, desc=f'{source_region} to {target_region} connectivity using {metric}'))

    conn_group_data = [item for sublist in results for item in sublist]
    all_pairs_df = pd.concat(conn_group_data)
    all_pairs_df.to_csv(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{analysis_evs[0]}_df_raw', index=False)

In [ ]:
smf.mixedlm('conn ~ 1 + zrpe', 
all_pairs_df, groups='participant').fit().summary()

In [ ]:
sns.lmplot(data=all_pairs_df, x='zrpe', y='conn', scatter=False, hue='participant')

In [ ]:
# %%capture
# pairs = (['HPC', 'ACC'], ['HPC', 'dmPFC'], ['HPC', 'dlPFC'] )

# # ['AMY', 'OFC'], ['dmPFC', 'OFC'], ['dlPFC', 'OFC']

# for pair in pairs:
#     source_region = pair[0]
#     target_region = pair[1]
    
#     # initialize the storage list 
#     conn_group_data = []

#     # iterate through subjects
#     for subj_id in tqdm(subj_ids, desc=f'{source_region} to {target_region} connectivity using {metric}'):
#         # display(f'Computing granger causality for {pair} in {subj_id}')
#         for event in analysis_evs:
#             loadpath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}'
#             epochs_reref = mne.read_epochs(f'{loadpath}/neural/{day}/{event}-epo.fif', preload=True) 

#             # Get electrode df 
#             elec_df = pd.read_csv(f'{loadpath}/Day1_reref_elec_df')
#             seed_target_df = oscillation_utils.make_seed_target_df(elec_df, epochs_reref, source_region, target_region)
            
#             # For within-region connectivity, let's drop elements that are the same
#             if len(seed_target_df) == 0:
#                 continue
                
#             # for cond in conditions: 
#             for hemi in ['l', 'r']:
#                 # first determine if ipsilateral connectivity is even possible; if not, move on. 
#                 # note: edit this if you want to compute contralateral connectivity 
#                 if hemi not in seed_target_df.index.tolist():
#                     continue
#                 else:
#                     seed_to_target = seed_target_indices(
#                         seed_target_df['seed'][hemi],
#                         seed_target_df['target'][hemi])
                
#                 pwise = run_surr_psi_time(epochs_reref.copy(),
#                                             indices = seed_to_target,
#                                             band = freq_dict[band1],
#                                             freqs = freqs[(freqs >= freq_dict[band1][0]) & (freqs <= freq_dict[band1][1])],
#                                             n_cycles = n_cycles[(freqs >= freq_dict[band1][0]) & (freqs <= freq_dict[band1][1])],
#                                             buf_ms = buf_ms,
#                                             n_surr = n_surr,
#                                             surr_method = 'swap_time_blocks')

#                 pwise_dfs = []

#                 npairs = len(seed_to_target[0])
                    
#                 for pair in range(npairs): 
#                     if epochs_reref.ch_names[seed_to_target[0][pair]] == epochs_reref.ch_names[seed_to_target[1][pair]]:
#                         # skip any same-channel connectivity
#                         continue
#                     pwise_df = pd.DataFrame(columns=['participant', 'roi1', 'roi2', 'hemi', 'pair_label', 'metric', 'event', 'conn'])
#                     pwise_df['conn'] = pwise[:, pair] 
#                     pwise_df['participant'] = subj_id
#                     pwise_df['age'] = subj_df[subj_df.MSSMCode==subj_id].Age.values[0]
#                     pwise_df['sex'] = subj_df[subj_df.MSSMCode==subj_id].Sex.str.strip().values[0]
#                     pwise_df['roi1'] = source_region
#                     pwise_df['roi2'] = target_region
#                     pwise_df['hemi'] = hemi 
#                     pwise_df['metric'] = metric
#                     pwise_df['event'] = event
#                     pwise_df['band0'] = band1
#                     pwise_df['band1'] =  band2
                    
#                     pwise_df['pair_label'] = f'{epochs_reref.ch_names[seed_to_target[0][pair]]}:{epochs_reref.ch_names[seed_to_target[1][pair]]}'

#                     if event in ['gamble_start', 'feedback_start', 'baseline_start']:
#                         pwise_df['trials'] = np.arange(1, learn_df[learn_df.participant==subj_id].shape[0]+1)
#                         pwise_df = pwise_df.merge(learn_df[(learn_df.participant==subj_id)], on=['participant', 'trials'])
#                     elif event == 'cue_on':
#                         memory_df = combined_df[(combined_df.condition=='Day1') & (combined_df.participant==subj_id)].dropna(subset=['trials_mem']).sort_values(by=['trials_mem'])[['participant', 'trials_mem', 'hits']]
#                         memory_df.rename(columns={'trials_mem':'trials'}, inplace=True)
#                         pwise_df['trials'] = np.arange(1, memory_df.shape[0]+1)
#                         pwise_df = pwise_df.merge(memory_df, on=['participant', 'trials'])

#                     # aggregate
#                     pwise_dfs.append(pwise_df)
#                 pwise_dfs = pd.concat(pwise_dfs)
#                 conn_group_data.append(pwise_dfs)

#     all_pairs_df = pd.concat(conn_group_data)
#     all_pairs_df.to_csv(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_df', index=False)

In [ ]:
savepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/Connectivity/scratch/{metric}'

In [ ]:
band1 = 'theta'
band2 = 'theta'
metric = 'psi_time'
source_region = 'AMY'
event = 'feedback_start'
test_dfs = []
for roi in ['ACC', 'dmPFC', 'dlPFC', 'OFC']:
    test_df = pd.read_csv(f'{savepath}/{source_region}_{roi}_{metric}_{band1}_{band2}_{event}_df_surr_trimmed')
    test_df=test_df[['participant', 'trials', 'pair_label', 'roi2', 'conn', 'hemi', 'reward']].rename(columns={'trials':'trials_gamble'}).merge(combined_df, on=['participant', 'trials_gamble'])
    test_dfs.append(test_df)


In [ ]:
test_dfs = pd.concat(test_dfs)
test_dfs = test_dfs[test_dfs.condition=='Day1']

In [ ]:
sns.barplot(data=test_dfs, x='roi2', y='conn')

In [ ]:
test_dfs[test_dfs.conn>4]

In [ ]:
test_dfs['label1'] = test_dfs['pair_label'].str.split(':').str[0]
test_dfs['label2'] = test_dfs['pair_label'].str.split(':').str[1]
test_dfs['label1_power'] = np.nan
test_dfs['label2_power'] = np.nan


In [ ]:
test_dfs.reset_index(drop=True, inplace=True)

In [ ]:
# Load in the theta power for each electrode involved in the PSI analysis 
# LOAD the data
analysis_evs = ['feedback_start']
chan_dfs = [] 
glms = []
elec_counter = 0 
block = [0.0, 1.5]
for subj_id in subj_ids:
    print(f'Loading subj {subj_id}')
    day = 'Day1'
    filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/TFR'
    elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/{day}_reref_elec_df')


    for event in analysis_evs:
        for chan in elec_df.label.unique():
            
            # for chan in tqdm(ch_names, ascii=True, desc='Loading TFR dataframes'): 
            chan_df = pd.read_csv(f'{filepath}/{event}/dfs/{chan}_time_averaged_df_time{block[0]}-{block[1]}.csv')


            chan_df['unique_elec_id'] = elec_counter
            chan_df['age'] = subj_df[subj_df.MSSMCode==subj_id].Age.values[0]
            chan_df['sex'] = subj_df[subj_df.MSSMCode==subj_id].Sex.str.strip().values[0]
            chan_df['hemi'] = elec_df[elec_df.label==chan].hemisphere.str.lower().values[0]


            # # learning only: 
            # chan_df = chan_df[chan_df.gamble_rt>0.3]
            # chan_df = chan_df[chan_df.gamble_rt<10]

            # with memory
            # memory_df = combined_df[(combined_df.condition=='Day1') & (combined_df.participant==chan_df.participant.unique()[0])].dropna(subset=['trials_gamble']).sort_values(by=['trials_gamble'])[['participant', 'trials_gamble', 'hits', 'recog_rt', 'zrpe', 'zQ', 'zRT_rpe', 'preward', 'ztrials_since_reversal', 'zprior_rpe']]
            # memory_df.rename(columns={'trials_gamble':'trials'}, inplace=True)
            # memory_df['phit'] = memory_df.apply(lambda x: 0.5 if x.hits==True else -0.5, axis=1)
            # chan_df = chan_df.merge(test_df, on=['participant', 'trials'])

            # chan_df = chan_df[chan_df.recog_rt>0.3]
            # chan_df = chan_df[chan_df.recog_rt<10]
            chan_df.rename(columns={'trials':'trials_gamble'}, inplace=True)
            
            if chan in test_dfs[test_dfs.participant==chan_df.participant.unique()[0]].label1.unique():
                chan_df.rename(columns={'channel':'label1',
                                        'tfr': 'label1_power'}, inplace=True)

                df_temp = test_dfs.merge(chan_df.loc[(chan_df.fband=='high_theta'), ['participant', 'trials_gamble', 'label1', 'label1_power']],
                                                            on=['participant', 'trials_gamble', 'label1'], how='left', suffixes=('', '_from_df2')) 

                test_dfs['label1_power'] = test_dfs['label1_power'].combine_first(df_temp['label1_power_from_df2'])  

            if chan in test_dfs[test_dfs.participant==chan_df.participant.unique()[0]].label2.unique():
                chan_df.rename(columns={'channel':'label2',
                                        'tfr': 'label2_power'}, inplace=True)
                
                df_temp = test_dfs.merge(chan_df.loc[(chan_df.fband=='high_theta'), ['participant', 'trials_gamble', 'label2', 'label2_power']],
                                                            on=['participant', 'trials_gamble', 'label2'], how='left', suffixes=('', '_from_df2')) 

                test_dfs['label2_power'] = test_dfs['label2_power'].combine_first(df_temp['label2_power_from_df2'])    
            else:
                continue
            elec_counter +=1

#             chan_dfs.append(chan_df)

# chan_dfs_feedback = pd.concat(chan_dfs)
# chan_dfs_feedback['zAge'] =(chan_dfs_feedback.age - np.nanmean(chan_dfs_feedback.age)) / (2*np.nanstd(chan_dfs_feedback.age))



In [ ]:
sns.regplot(data=test_dfs, x='label1_power', y='conn', scatter=False)

In [ ]:
import statsmodels.formula.api as smf
smf.mixedlm('conn ~ 1 + label1_power + label2_power + zrpe*phit',
            test_dfs[(np.abs(test_dfs.label1_power)<5) & (np.abs(test_dfs.label2_power)<5)], 
            groups='participant').fit().summary()

In [ ]:
smf.mixedlm('conn ~ 1 + rpe*hits', test_df, groups='pair_label').fit().summary()

2/24/2025: Both OFC and ACC show PSI connectivity with HPC that is up-regulated by RPE:memory

## Directed Connectivity (PSI or Granger) - computed over epochs

In [ ]:
analysis_evs = ['cue_start']

In [ ]:
metric = 'psi'
freqs = np.logspace(*np.log10([2, 200]), num=30)
n_cycles = np.floor(np.logspace(*np.log10([3, 10]), num=30))

freq_dict = {'theta':[4, 8],
            'alpha':[8, 13],
            'beta': [13, 30], 
            'hfa': [70, 200]}

# freq_dict = {'high_theta':[4, 8]}

band1 = 'theta'
band2 = 'theta'
day = 'Day1'

# band1 = 'theta'
# band2 = 'theta'

buf_ms = 1000
n_surr = 100

savepath = f'{base_dir}/scratch/MemoryBandit/{metric}'
if not os.path.exists(savepath):
    os.makedirs(savepath)

In [ ]:
# def make_seed_target_df(elec_df, epochs, source_roi, target_roi): 
    
#     """
#     Create arrays of indices for mapping electrodes for connectivity analyses. Use the Epoch
#     ch_names list itself to find the index of the electrode within the mne object 
    
#     """
    
#     seed_target_df = pd.DataFrame(columns=['seed', 'target'], index=['l', 'r'])

#     for hemi in ['l', 'r']:
#         source_ix = elec_df[(elec_df.hemisphere.str.lower()==hemi) & (elec_df.salman_region==source_roi)].label.values
#         target_ix = elec_df[(elec_df.hemisphere.str.lower()==hemi) & (elec_df.salman_region==target_roi)].label.values

#         if (len(source_ix) == 0) | (len(target_ix)==0):
#             seed_target_df['seed'][hemi] = []
#             seed_target_df['target'][hemi] = []
#         else:   
#             seed_target_df['seed'][hemi] = mne.pick_channels(epochs.ch_names, source_ix)

#             seed_target_df['target'][hemi] = mne.pick_channels(epochs.ch_names, target_ix)

#     seed_target_df = seed_target_df[
#                 (seed_target_df['seed'].map(lambda d: len(d) > 0)) & (seed_target_df['target'].map(lambda d: len(d) > 0))]

#     return seed_target_df
    

12/18/2025: I want to split the encoding period into a first half (e.g. low ACC activity relating to RPE coding) and the second half (e.g. high ACC activity relating to RPE coding) and see if this change results in some gross change in information flow to HPC 

In [ ]:
epochs_reref._data.shape

In [ ]:
%%capture

# pairs = (['AMY', 'HPC'], ['AMY', 'OFC'], ['AMY', 'dmPFC'], ['AMY', 'ACC'])
# ['HPC', 'OFC'], , ['ACC', 'OFC'], ['dmPFC','OFC'], ['dlPFC', 'OFC']
pairs = (['ACC', 'HPC'], ['OFC', 'HPC'], ['dmPFC', 'HPC'], ['dlPFC', 'HPC'])

    
            
# initialize the storage list 
conn_dfs = []

# iterate through subjects
for subj_id in subj_ids:
    for event in analysis_evs:
        loadpath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}'
        epochs_reref = mne.read_epochs(f'{loadpath}/neural/{day}/{event}-epo.fif', preload=True, verbose='ERROR'); 
        
        # Get electrode df 
        elec_df = pd.read_csv(f'{loadpath}/Day1_reref_elec_df')

        for pair in pairs:

            source_region = pair[0]
            target_region = pair[1]


            seed_target_df = oscillation_utils.make_seed_target_df(elec_df, epochs_reref, source_region, target_region)
            
            if len(seed_target_df) == 0:
                continue
                
            # for cond in conditions:   
            for hemi in ['l', 'r']:
                # first determine if ipsilateral connectivity is even possible; if not, move on. 
                # note: edit this if you want to compute contralateral connectivity 
                if hemi not in seed_target_df.index.tolist():
                    continue
                else:
                    seed_to_target = seed_target_indices(
                        seed_target_df['seed'][hemi],
                        seed_target_df['target'][hemi])
            
            
                pwise = oscillation_utils.compute_connectivity(epochs_reref.copy(), 
                                               band = freq_dict[band1], 
                                               metric = metric, 
                                               indices = seed_to_target, 
                                               freqs = freqs, 
                                               n_cycles = n_cycles,
                                               buf_ms = buf_ms, 
                                               n_surr=n_surr,
                                               avg_over_dim='time',
                                               band1 = freq_dict[band2],
                                               parallelize=True,
                                               gc_n_lags=5)

                if metric == 'granger':
                    npairs = 1
                else:
                    npairs = len(seed_to_target[0])
                    pair_labels = [] 
                    for pair_ix in range(npairs):
                        pair_labels.append(f'{epochs_reref.ch_names[seed_to_target[0][pair_ix]]}:{epochs_reref.ch_names[seed_to_target[1][pair_ix]]}')


                row_metadata = pd.DataFrame({'conn': np.squeeze(pwise)}) 

                # row_metadata['pair_labels'] = pair_labels
                row_metadata['participant'] = subj_id
                row_metadata['age'] = subj_df[subj_df.MSSMCode==subj_id].Age.values[0]
                row_metadata['sex'] = subj_df[subj_df.MSSMCode==subj_id].Sex.str.strip().values[0]
                row_metadata['roi1'] = source_region
                row_metadata['roi2'] = target_region
                row_metadata['hemi'] = hemi
                row_metadata['metric'] = metric
                row_metadata['event'] = event
                row_metadata['band0'] = band1
                row_metadata['band1'] = band2
                # row_metadata['trial'] = 
                
                conn_dfs.append(row_metadata)
                # conn_group_data.append(pwise)
                # conn_group_metadata.append(row_metadata)
            
conn_group_data = pd.concat(conn_dfs)
conn_group_data.to_csv(f'{savepath}/{metric}_{band1}_{band2}_{event}_df_Z', index=False)
    
    # # ACTION: SWAP
    # np.save(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_Z.npy', conn_group_data)
    # conn_group_metadata.to_csv(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_df_Z', index=False)
    

In [ ]:
conn_group_data.to_csv(f'{savepath}/{metric}_{band1}_{band2}_{event}_df_Z', index=False)


In [ ]:
conn_group_data = pd.concat(conn_dfs)


In [ ]:
conn_group_data

In [ ]:
subj_id

In [ ]:
f'{savepath}/{metric}_{band1}_{band2}_{event}_df_Z'

Separate out the conditions we want to analyze using PSI. Previously I did positive vs. negative rPE

2/18/25: 

For feedback, this can just be hits vs misses 

For cue, this should be the 4 types of memory responses

In [ ]:
conn_group_data['trials'] = (np.arange(80).tolist()) * int(conn_group_data.shape[0]/80)

In [ ]:
test_df = conn_group_data.merge(learn_df[['participant', 'rpe_cat', 'trials', 'rpe']], on = ['participant', 'trials'])

In [ ]:
sns.barplot(data=test_df, x = 'roi1', y = 'conn', hue = 'rpe_cat', errorbar='se', hue_order=['neg', 'neutral', 'pos'])

In [ ]:
test = oscillation_utils.compute_gc_tr(mne_data=epochs_reref.copy(), 
                    band=freq_dict['theta'],
                    indices=seed_to_target, 
                    freqs=freqs[(freqs>=freq_dict['theta'][0]) & (freqs<=freq_dict['theta'][1])], 
                    n_cycles=n_cycles[(freqs>=freq_dict['theta'][0]) & (freqs<=freq_dict['theta'][1])],
                    rank=None, 
                    gc_n_lags=5, 
                    buf_ms=buf_ms, 
                    avg_over_dim='time')

In [ ]:
plt.plot(test)

In [ ]:
pwise

In [ ]:
# pairs = (['AMY', 'HPC'], ['AMY', 'OFC'], ['AMY', 'dmPFC'], ['AMY', 'ACC'])
# ['HPC', 'OFC'], , ['ACC', 'OFC'], ['dmPFC','OFC'], ['dlPFC', 'OFC']
pairs = (['ACC', 'OFC'], ['ACC', 'dmPFC'], ['ACC', 'dlPFC'])

for pair in pairs:
    source_region = pair[0]
    target_region = pair[1]
    
    # initialize the storage list 
    conn_group_data = []
    conn_group_metadata = []
    
    # iterate through subjects
    for subj_id in tqdm(subj_ids, desc=f'{source_region} to {target_region} connectivity using {metric}'): 
        for event in analysis_evs:
            loadpath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}'
            epochs_reref = mne.read_epochs(f'{loadpath}/neural/{day}/{event}-epo.fif', preload=True, verbose='ERROR'); 
            
            # Get electrode df 
            elec_df = pd.read_csv(f'{loadpath}/Day1_reref_elec_df')

            seed_target_df = oscillation_utils.make_seed_target_df(elec_df, epochs_reref, source_region, target_region)
            
            if len(seed_target_df) == 0:
                continue
                
            # for cond in conditions: 
            for hemi in ['l', 'r']:
                # first determine if ipsilateral connectivity is even possible; if not, move on. 
                # note: edit this if you want to compute contralateral connectivity 
                if hemi not in seed_target_df.index.tolist():
                    continue
                else:
                    seed_to_target = seed_target_indices(
                        seed_target_df['seed'][hemi],
                        seed_target_df['target'][hemi])
            
            
                pwise = oscillation_utils.compute_connectivity(epochs_reref.copy(), 
                                               band = freq_dict[band1], 
                                               metric = metric, 
                                               indices = seed_to_target, 
                                               freqs = freqs, 
                                               n_cycles = n_cycles,
                                               buf_ms = buf_ms, 
                                               n_surr=n_surr,
                                               avg_over_dim='epochs',
                                               surr_method='swap_time_blocks',
                                               band1 = freq_dict[band2],
                                               parallelize=True,
                                               gc_n_lags=20)

                npairs = len(seed_to_target[0])
                pair_labels = [] 
                for pair_ix in range(npairs):
                    pair_labels.append(f'{epochs_reref.ch_names[seed_to_target[0][pair_ix]]}:{epochs_reref.ch_names[seed_to_target[1][pair_ix]]}')


                row_metadata = pd.DataFrame({
                    'pair_labels': pair_labels,
                    'participant': subj_id,
                    'age': subj_df[subj_df.MSSMCode==subj_id].Age.values[0], 
                    'sex': subj_df[subj_df.MSSMCode==subj_id].Sex.str.strip().values[0],
                    'roi1': source_region, 
                    'roi2': target_region, 
                    'hemi': hemi, 
                    'metric': metric,
                    'event': event,
                    'band0': band1, 
                    'band1': band2,
                    'cond':'hit'}) # ACTION: SWAP

                conn_group_data.append(pwise)
                conn_group_metadata.append(row_metadata)
            
    conn_group_data = np.vstack(conn_group_data)
    conn_group_metadata = pd.concat(conn_group_metadata)
    
    # ACTION: SWAP
    np.save(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_all.npy', conn_group_data)
    conn_group_metadata.to_csv(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_all_df', index=False)
    
    

In [ ]:
# Compare different TARGET REGIONS for a single condition
# from scipy.ndimage import uniform_filter1d
source_region = 'ACC'
event = 'feedback_start'
band1 = 'theta'
band2 = 'theta'
condition = 'all'
metric = 'psi'

savepath = f'/Volumes/T7 Shield/scratch/MemoryBandit/{metric}'
plot_dir = '/Users/salmanqasim/Documents/Projects/MemBanditsEMU/Plots/MemoryBandit-Connectivity'
os.makedirs(plot_dir, exist_ok=True)

# Define target regions to compare
target_regions = ['OFC', 'dlPFC', 'dmPFC']  # Modify this list as needed
colors = {'ACC': 'blue', 'OFC': 'green', 'dlPFC': 'red', 'dmPFC': 'purple', 'AMY': 'orange', 'HPC': 'cyan'}

fig, ax = plt.subplots(1, 1, figsize=(6, 4), dpi=300)

# Store data for pairwise statistical comparisons
region_data = {}

for target_region in target_regions:
    try:
        conn_group_data = np.load(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_all.npy')
        meta_data = pd.read_csv(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_all_df')
        meta_data.hemi = meta_data.hemi.str.lower()
        hemi_index = meta_data[meta_data.hemi.isin(['l', 'r'])].index.values
        
        if len(hemi_index) == 0:
            print(f"No valid electrodes for {target_region}")
            continue
            
        print(f"{target_region}: {len(hemi_index)} electrode pairs")
        
        # Store for statistical comparison
        smoothed_data = conn_group_data[hemi_index, :]
        # smoothed_data = np.array([uniform_filter1d(conn_group_data[idx, :], size=100, mode='nearest') 
        #                       for idx in hemi_index])
        region_data[target_region] = smoothed_data
        
        # Calculate mean and SEM
        stderr = np.nanstd(smoothed_data[hemi_index, :], axis=0) / np.sqrt(conn_group_data[hemi_index, :].shape[0])
        mean_data = np.nanmean(smoothed_data[hemi_index, :], axis=0)
        
        # Set up time axis
        times = np.linspace(0, 1500, smoothed_data[hemi_index, :].shape[1])
        
        # Plot
        color = colors.get(target_region, 'gray')
        ax.plot(times, mean_data, color=color, label=f'{source_region}-{target_region}')
        ax.fill_between(times, mean_data - stderr, mean_data + stderr, color=color, alpha=0.3)
        
    except FileNotFoundError:
        print(f"Data not found for {source_region}-{target_region}")
        continue

# Compare all regions to HPC-ACC (reference_region)
reference_region = 'ACC'
region_names = list(region_data.keys())
if reference_region in region_names and len(region_names) >= 2:
    print(f"\n--- Comparisons to {source_region}-{reference_region} ---")
    for r in region_names:
        if r != reference_region:
            T_obs, clusters, cluster_p_values, H0 = mne.stats.permutation_cluster_test(
                [region_data[reference_region], region_data[r]], 
                n_permutations=1000, 
                out_type='mask', 
                seed=2
            )
            sig_clusters = [p for p in cluster_p_values if p <= 0.05]
            print(f"{reference_region} vs {r}: {len(sig_clusters)} significant clusters (p <= 0.05)")
            if len(sig_clusters) > 0:
                print(f"  p-values: {[f'{p:.3f}' for p in cluster_p_values if p <= 0.05]}")

ax.axhline(0, color='k', linestyle='--', linewidth=0.5)
ax.legend(loc='best')
sns.despine()
ax.set(xlabel='time (ms)', ylabel=f'{metric} (z)', title=f'{source_region} connectivity: {condition.replace("_index", "")}')
plt.savefig(f'{plot_dir}/{metric}_{source_region}_regions_comparison_{band1}_{event}_{condition}.pdf', dpi=300)

In [ ]:
tfr_data.ch_names

In [ ]:
# Load the baeline-corrected TFR data: 

subj_id = 'MS030'
event = 'feedback_start'
base_dir = '/Volumes/T7 Shield' # this is the root directory for most un-archived data and results

filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/TFR'
tfr_data = mne.time_frequency.read_tfrs(f'{filepath}/{event}-tfr.h5')

source_region = 'ACC'
target_region = 'HPC'

# Get electrode df 
loadpath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}'

elec_df = pd.read_csv(f'{loadpath}/Day1_reref_elec_df')
seed_target_df = oscillation_utils.make_seed_target_df(elec_df, tfr_data, source_region, target_region)


pos_rpe_index = learn_df[(learn_df.participant==subj_id) & (learn_df.rpe_cat=='pos')].trials.values.astype(int) - 1
neg_rpe_index = learn_df[(learn_df.participant==subj_id) & (learn_df.rpe_cat=='neg')].trials.values.astype(int) - 1
neutral_rpe_index = learn_df[(learn_df.participant==subj_id) & (learn_df.rpe_cat=='neutral')].trials.values.astype(int) - 1

# for cond in conditions: 
for hemi in ['l', 'r']:
    # first determine if ipsilateral connectivity is even possible; if not, move on. 
    # note: edit this if you want to compute contralateral connectivity 
    if hemi not in seed_target_df.index.tolist():
        continue
    else:
        seed_to_target = seed_target_indices(
            seed_target_df['seed'][hemi],
            seed_target_df['target'][hemi])


    freq_tes_pos = oscillation_utils.compute_te(
        tfr_data.data[pos_rpe_index, :, :, :],
        indices=seed_to_target,
        buf_ms= 0,
        te_k=1,
        surr_method='swap_time_blocks',
        parallelize=True,
        n_surr=100,
        return_freqs=True,
        net=True,
        sfreq=tfr_data.sfreq,
        freqs=tfr_data.freqs)

    freq_tes_neg = oscillation_utils.compute_te(
        tfr_data.data[neg_rpe_index, :, :, :],
        indices=seed_to_target,
        buf_ms= 0,
        te_k=1,
        surr_method='swap_time_blocks',
        parallelize=True,
        n_surr=100,
        return_freqs=True,
        net=True,
        sfreq=tfr_data.sfreq,
        freqs=tfr_data.freqs)

    freq_tes_neutral = oscillation_utils.compute_te(
        tfr_data.data[neutral_rpe_index, :, :, :],
        indices=seed_to_target,
        buf_ms= 0,
        te_k=1,
        surr_method='swap_time_blocks',
        parallelize=True,
        n_surr=100,
        return_freqs=True,
        net=True,
        sfreq=tfr_data.sfreq,
        freqs=tfr_data.freqs)
        
    # Store the results for a loop across source regions and subjects
    

In [ ]:
plt.plot(freq_tes[1], np.mean(freq_tes_pos[0], axis=0), color='green')
plt.plot(freq_tes[1], np.mean(freq_tes_neg[0], axis=0), color='red')

In [ ]:
plt.plot(freq_tes[1], np.mean(freq_tes[0], axis=0))

In [ ]:
n_times

In [ ]:
tfr_data._data.shape

In [ ]:
freqs = np.logspace(*np.log10([2, 200]), num=30)

plt.plot(freqs, freq_tes)

In [ ]:
np.load(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_all.npy')

In [ ]:
f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}'

In [ ]:
conn_group_data

In [ ]:

# pairs = (['AMY', 'HPC'], ['AMY', 'OFC'], ['AMY', 'dmPFC'], ['AMY', 'ACC'])
# ['HPC', 'OFC'], , ['ACC', 'OFC'], ['dmPFC','OFC'], ['dlPFC', 'OFC']
pairs = (['HPC', 'OFC'], ['HPC', 'dmPFC'], ['HPC', 'dlPFC'], ['HPC', 'ACC'])
         
for pair in pairs:
    source_region = pair[0]
    target_region = pair[1]
    
    # initialize the storage list 
    conn_group_data = []
    conn_group_metadata = []
    
    # iterate through subjects
    for subj_id in tqdm(subj_ids, desc=f'{source_region} to {target_region} connectivity using {metric}'): 
        for event in analysis_evs:
            loadpath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}'
            epochs_reref = mne.read_epochs(f'{loadpath}/neural/{day}/{event}-epo.fif', preload=True); 
            
            # MEMORY BASED PARSING
            if event == 'feedback_start':
                memory_df = combined_df[(combined_df.condition=='Day1') & (combined_df.participant==subj_id)].dropna(subset=['trials_gamble']).sort_values(by=['trials_gamble'])[['participant', 'trials_gamble', 'hits', 'rpe']]
                memory_df.rename(columns={'trials_gamble':'trials'}, inplace=True)
                memory_df['phit'] = memory_df.apply(lambda x: 1 if x.hits==True else 0, axis=1)

                pos_rpe_threshold = np.percentile(combined_df[(combined_df.condition=='Day1') & (combined_df.participant==subj_id)].dropna(subset=['trials_mem']).rpe, 66)
                neg_rpe_threshold = np.percentile(combined_df[(combined_df.condition=='Day1') & (combined_df.participant==subj_id)].dropna(subset=['trials_mem']).rpe, 33)
                pos_rpe_index = memory_df[(memory_df.rpe>pos_rpe_threshold)].trials.values.astype(int) - 1
                neg_rpe_index = memory_df[(memory_df.rpe<neg_rpe_threshold)].trials.values.astype(int) - 1

                nrew_index = memory_df[(memory_df.rpe<0)].trials.values.astype(int) - 1
                miss_index = memory_df[memory_df.hits==False].trials.values.astype(int) - 1
                miss_rew_index = memory_df[(memory_df.hits==False) & (memory_df.rpe>0)].trials.values.astype(int) - 1

            elif event == 'cue_on':
                memory_df = combined_df[(combined_df.condition=='Day1') & (combined_df.participant==subj_id)].dropna(subset=['trials_mem']).sort_values(by=['trials_mem'])[['participant', 'trials_mem', 'hits', 'rpe']]
                memory_df.rename(columns={'trials_mem':'trials'}, inplace=True)
                memory_df['phit'] = memory_df.apply(lambda x: 1 if x.hits==True else 0, axis=1)

                miss_index = memory_df[memory_df.hits==False].trials.values.astype(int) - 1
                nrew_index = memory_df[(memory_df.rpe<0)].trials.values.astype(int) - 1
            # trunc_epochs = tfr_data[epochs_to_analyze.astype(int)]

            
            # # RPE BASED PARSING
            # pos_rpe_index = np.where(learn_df[learn_df.participant==subj_id].rpe>0)[0]           

            # Get electrode df 
            elec_df = pd.read_csv(f'{loadpath}/Day1_reref_elec_df')

            seed_target_df = oscillation_utils.make_seed_target_df(elec_df, epochs_reref, source_region, target_region)

            if len(seed_target_df) == 0:
                continue
                
            # for cond in conditions: 
            for hemi in ['l', 'r']:
                # first determine if ipsilateral connectivity is even possible; if not, move on. 
                # note: edit this if you want to compute contralateral connectivity 
                if hemi not in seed_target_df.index.tolist():
                    continue
                else:
                    seed_to_target = seed_target_indices(
                        seed_target_df['seed'][hemi],
                        seed_target_df['target'][hemi])
            
            
                pwise = oscillation_utils.compute_connectivity(epochs_reref.copy()[neg_rpe_index], # ACTION: SWAP
                                                   band = freq_dict[band1], 
                                                   metric = metric, 
                                                   indices = seed_to_target, 
                                                   freqs = freqs, 
                                                   n_cycles = n_cycles,
                                                   buf_ms = buf_ms, 
                                                   n_surr=n_surr,
                                                   avg_over_dim='epochs',
                                                   surr_method = 'swap_time_blocks',
                                                   band1 = freq_dict[band2],
                                                   parallelize=True)

                npairs = len(seed_to_target[0])
                pair_labels = [] 
                for pair_ix in range(npairs):
                    pair_labels.append(f'{epochs_reref.ch_names[seed_to_target[0][pair_ix]]}:{epochs_reref.ch_names[seed_to_target[1][pair_ix]]}')


                row_metadata = pd.DataFrame({
                    'pair_labels': pair_labels,
                    'participant': subj_id,
                    'age': subj_df[subj_df.MSSMCode==subj_id].Age.values[0], 
                    'sex': subj_df[subj_df.MSSMCode==subj_id].Sex.str.strip().values[0],
                    'roi1': source_region, 
                    'roi2': target_region, 
                    'hemi': hemi, 
                    'metric': metric,
                    'event': event,
                    'band0': band1, 
                    'band1': band2,
                    'cond':'miss'}) # ACTION: SWAP

                conn_group_data.append(pwise)
                conn_group_metadata.append(row_metadata)
            
    conn_group_data = np.vstack(conn_group_data)
    conn_group_metadata = pd.concat(conn_group_metadata)
    
    # ACTION: SWAP
    np.save(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_neg_rpe_index.npy', conn_group_data)
    conn_group_metadata.to_csv(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_neg_rpe_index_df', index=False)
    

In [ ]:
# For just cue_on, do false alarms and correct rejections: 

# pairs = (['AMY', 'HPC'], ['AMY', 'OFC'], ['AMY', 'dmPFC'], ['AMY', 'ACC'])
# ['HPC', 'OFC'], , ['ACC', 'OFC'], ['dmPFC','OFC'], ['dlPFC', 'OFC']
pairs = (['HPC', 'OFC'], ['HPC', 'dmPFC'], ['HPC', 'dlPFC'], ['HPC', 'ACC'])
         
for pair in pairs:
    source_region = pair[0]
    target_region = pair[1]
    
    # initialize the storage list 
    conn_group_data = []
    conn_group_metadata = []
    
    # iterate through subjects
    for subj_id in subj_ids:
        for event in analysis_evs:
            loadpath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}'
            epochs_reref = mne.read_epochs(f'{loadpath}/neural/{day}/{event}-epo.fif', preload=True); 
            
            # MEMORY BASED PARSING
            if event in ['cue_on', 'recog_time']:
                memory_df = combined_df[(combined_df.condition=='Day1') & (combined_df.participant==subj_id)].dropna(subset=['trials_mem']).sort_values(by=['trials_mem'])[['participant', 'trials_mem', 'memory_outcome']]
                memory_df.rename(columns={'trials_mem':'trials'}, inplace=True)

                fa_index = memory_df[memory_df.memory_outcome=='false_alarms'].trials.values.astype(int) - 1

            # Get electrode df 
            elec_df = pd.read_csv(f'{loadpath}/Day1_reref_elec_df')

            seed_target_df = oscillation_utils.make_seed_target_df(elec_df, epochs_reref, source_region, target_region)

            if len(seed_target_df) == 0:
                continue
                
            # for cond in conditions: 
            for hemi in ['l', 'r']:
                # first determine if ipsilateral connectivity is even possible; if not, move on. 
                # note: edit this if you want to compute contralateral connectivity 
                if hemi not in seed_target_df.index.tolist():
                    continue
                else:
                    seed_to_target = seed_target_indices(
                        seed_target_df['seed'][hemi],
                        seed_target_df['target'][hemi])
            
            
                pwise = oscillation_utils.compute_connectivity(epochs_reref.copy()[fa_index], # ACTION: SWAP
                                                   band = freq_dict[band1], 
                                                   metric = metric, 
                                                   indices = seed_to_target, 
                                                   freqs = freqs, 
                                                   n_cycles = n_cycles,
                                                   buf_ms = buf_ms, 
                                                   n_surr=n_surr,
                                                   avg_over_dim='epochs',
                                                   surr_method = 'swap_time_blocks',
                                                   band1 = freq_dict[band2],
                                                   parallelize=True)

                npairs = len(seed_to_target[0])
                pair_labels = [] 
                for pair_ix in range(npairs):
                    pair_labels.append(f'{epochs_reref.ch_names[seed_to_target[0][pair_ix]]}:{epochs_reref.ch_names[seed_to_target[1][pair_ix]]}')


                row_metadata = pd.DataFrame({
                    'pair_labels': pair_labels,
                    'participant': subj_id,
                    'age': subj_df[subj_df.MSSMCode==subj_id].Age.values[0], 
                    'sex': subj_df[subj_df.MSSMCode==subj_id].Sex.str.strip().values[0],
                    'roi1': source_region, 
                    'roi2': target_region, 
                    'hemi': hemi, 
                    'metric': metric,
                    'event': event,
                    'band0': band1, 
                    'band1': band2,
                    'cond':'miss'}) # ACTION: SWAP

                conn_group_data.append(pwise)
                conn_group_metadata.append(row_metadata)
            
    conn_group_data = np.vstack(conn_group_data)
    conn_group_metadata = pd.concat(conn_group_metadata)
    
    # ACTION: SWAP
    np.save(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_fa_index.npy', conn_group_data)
    conn_group_metadata.to_csv(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_fa_index_df', index=False)

In [ ]:
# For just cue_on, do false alarms and correct rejections: 

# pairs = (['AMY', 'HPC'], ['AMY', 'OFC'], ['AMY', 'dmPFC'], ['AMY', 'ACC'])
# ['HPC', 'OFC'], , ['ACC', 'OFC'], ['dmPFC','OFC'], ['dlPFC', 'OFC']
pairs = (['HPC', 'OFC'], ['HPC', 'dmPFC'], ['HPC', 'dlPFC'], ['HPC', 'ACC'])
         
for pair in pairs:
    source_region = pair[0]
    target_region = pair[1]
    
    # initialize the storage list 
    conn_group_data = []
    conn_group_metadata = []
    
    # iterate through subjects
    for subj_id in subj_ids:
        for event in analysis_evs:
            loadpath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}'
            epochs_reref = mne.read_epochs(f'{loadpath}/neural/{day}/{event}-epo.fif', preload=True); 
            
            # MEMORY BASED PARSING
            if event in ['cue_on', 'recog_time']:
                memory_df = combined_df[(combined_df.condition=='Day1') & (combined_df.participant==subj_id)].dropna(subset=['trials_mem']).sort_values(by=['trials_mem'])[['participant', 'trials_mem', 'memory_outcome']]
                memory_df.rename(columns={'trials_mem':'trials'}, inplace=True)

                cr_index = memory_df[memory_df.memory_outcome=='correct_rejections'].trials.values.astype(int) - 1

            # Get electrode df 
            elec_df = pd.read_csv(f'{loadpath}/Day1_reref_elec_df')

            seed_target_df = oscillation_utils.make_seed_target_df(elec_df, epochs_reref, source_region, target_region)

            if len(seed_target_df) == 0:
                continue
                
            # for cond in conditions: 
            for hemi in ['l', 'r']:
                # first determine if ipsilateral connectivity is even possible; if not, move on. 
                # note: edit this if you want to compute contralateral connectivity 
                if hemi not in seed_target_df.index.tolist():
                    continue
                else:
                    seed_to_target = seed_target_indices(
                        seed_target_df['seed'][hemi],
                        seed_target_df['target'][hemi])
            
            
                pwise = oscillation_utils.compute_connectivity(epochs_reref.copy()[fa_index], # ACTION: SWAP
                                                   band = freq_dict[band1], 
                                                   metric = metric, 
                                                   indices = seed_to_target, 
                                                   freqs = freqs, 
                                                   n_cycles = n_cycles,
                                                   buf_ms = buf_ms, 
                                                   n_surr=n_surr,
                                                   avg_over_dim='epochs',
                                                   surr_method = 'swap_time_blocks',
                                                   band1 = freq_dict[band2],
                                                   parallelize=True)

                npairs = len(seed_to_target[0])
                pair_labels = [] 
                for pair_ix in range(npairs):
                    pair_labels.append(f'{epochs_reref.ch_names[seed_to_target[0][pair_ix]]}:{epochs_reref.ch_names[seed_to_target[1][pair_ix]]}')


                row_metadata = pd.DataFrame({
                    'pair_labels': pair_labels,
                    'participant': subj_id,
                    'age': subj_df[subj_df.MSSMCode==subj_id].Age.values[0], 
                    'sex': subj_df[subj_df.MSSMCode==subj_id].Sex.str.strip().values[0],
                    'roi1': source_region, 
                    'roi2': target_region, 
                    'hemi': hemi, 
                    'metric': metric,
                    'event': event,
                    'band0': band1, 
                    'band1': band2,
                    'cond':'miss'}) # ACTION: SWAP

                conn_group_data.append(pwise)
                conn_group_metadata.append(row_metadata)
            
    conn_group_data = np.vstack(conn_group_data)
    conn_group_metadata = pd.concat(conn_group_metadata)
    
    # ACTION: SWAP
    np.save(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_cr_index.npy', conn_group_data)
    conn_group_metadata.to_csv(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_cr_index_df', index=False)

In [ ]:
source_region = 'HPC'
event = 'feedback_start'
band1= 'theta'
band2= 'theta'
metric = 'psi'

savepath = f'/Volumes/T7 Shield/scratch/MemoryBandit/{metric}'

for target_region in ['OFC','dlPFC','ACC', 'dmPFC']:
    conn_group_data_pos = np.load(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_hit_index.npy') # pos_rpe
    conn_group_data_neg = np.load(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_miss_index.npy') # neg_rpe


    fig, ax = plt.subplots(1, 1, figsize=(5, 3), dpi=300)

    # Let's look at l vs r for HPC-OFC
    meta_data = pd.read_csv(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_hit_index_df') # pos_rpe
    meta_data.hemi = meta_data.hemi.str.lower()
    hemi_index = meta_data[meta_data.hemi.isin(['l', 'r'])].index.values
    print(len(hemi_index))


    T_obs, clusters, cluster_p_values, H0 = mne.stats.permutation_cluster_test([conn_group_data_pos[hemi_index, :], conn_group_data_neg[hemi_index, :]], 
                                                                                     n_permutations=1000, 
                                                                                     out_type='mask', seed=2)
    # T_obs, clusters, cluster_p_values, H0 = mne.stats.permutation_cluster_1samp_test(conn_group_data_neg, 
    #                                                                                  n_permutations=1000,
    #                                                                                  out_type='mask')
    print(cluster_p_values)
    stderr = np.nanstd(conn_group_data_pos[hemi_index, :], axis=0) / np.sqrt(conn_group_data_pos[hemi_index, :].shape[0])
    mean_data = np.nanmean(conn_group_data_pos[hemi_index, :], axis=0)

    if event == 'cue_on':
            times = np.linspace(0, 1000, conn_group_data_pos[hemi_index, :].shape[1])
    else:
        times = np.linspace(0, 1500, conn_group_data_pos[hemi_index, :].shape[1])
    # freqs = np.linspace(0, 500, conn_group_data_pos[:, :].shape[1])

    ax.plot(times, mean_data)
    ax.fill_between(times, mean_data - stderr, mean_data + stderr, color='green', alpha=0.3)

    stderr = np.nanstd(conn_group_data_neg[hemi_index, :], axis=0) / np.sqrt(conn_group_data_neg[hemi_index, :].shape[0])
    mean_data = np.nanmean(conn_group_data_neg[hemi_index, :], axis=0)


    ax.plot(times, mean_data)
    ax.fill_between(times, mean_data - stderr, mean_data + stderr, color='red', alpha=0.3)

    # ax.hlines(0, freqs[0], freqs[-1], 'k')

    for i_c, c in enumerate(clusters):
        c = c[0]
        if (cluster_p_values[i_c] <= 0.05) & ((times[c.stop - 1] - times[c.start]) > 125): # 166.6
            if target_region=='HPC':
                h = ax.axvspan(times[c.start], times[c.stop - 1], color="purple", alpha=0.3)
            elif target_region=='AMY':
                h = ax.axvspan(times[c.start], times[c.stop - 1], color="yellow", alpha=0.3)
            else: 
                h = ax.axvspan(times[c.start], times[c.stop - 1], color="k", alpha=0.3)
#         else:
#             ax.axvspan(times[c.start], times[c.stop - 1], color=(0.3, 0.3, 0.3), alpha=0.3)


    sns.despine()
    ax.set(xlabel='time (ms)', ylabel=f'{metric} (z)', ylim=[-0.75, 0.75])
    plt.savefig(f'{plot_dir}/{metric}_{source_region}_{target_region}_{band1}_{event}_memory_Z.pdf', dpi=300)


In [ ]:
T_obs.max()

In [ ]:
# Let's split the first half of the encoding period from the second half of the encoding period and compare the coherence computed across epochs
# between these two timepoints: 

source_region = 'HPC'
event = 'feedback_start'
band1= 'theta'
band2= 'theta'
metric = 'psi'

savepath = f'/Volumes/T7 Shield/scratch/MemoryBandit/{metric}'

first_half_pos = []
first_half_neg = []
second_half_pos = []
second_half_neg = []
for target_region in ['OFC', 'ACC', 'dlPFC', 'dmPFC']:
    conn_group_data_pos = np.load(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_pos_rpe_index.npy')
    conn_group_data_neg = np.load(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_neg_rpe_index.npy')
    length_window = conn_group_data_pos.shape[1]
    first_half_pos.append(conn_group_data_pos[:, :length_window//2].mean(axis=1))
    first_half_neg.append(conn_group_data_neg[:, :length_window//2].mean(axis=1))
    second_half_pos.append(conn_group_data_pos[:, length_window//2:].mean(axis=1))
    second_half_neg.append(conn_group_data_neg[:, length_window//2:].mean(axis=1))


first_half_pos = np.concatenate(first_half_pos)
first_half_neg = np.concatenate(first_half_neg)
second_half_pos = np.concatenate(second_half_pos)
second_half_neg = np.concatenate(second_half_neg)


print(ttest_rel(first_half_pos, first_half_neg))
print(ttest_rel(second_half_pos, second_half_neg))
print(ttest_rel(first_half_pos-first_half_neg, second_half_pos-second_half_neg))

fig, ax = plt.subplots(1, 2, figsize=(2, 3), dpi=300, sharey=True)

sns.barplot(data=pd.DataFrame({'first_half':first_half_pos - first_half_neg}), ax=ax[0], errorbar='se')
sns.barplot(data=pd.DataFrame({'second_half':second_half_pos - second_half_neg}), ax=ax[1], errorbar='se')

# Add a stripplot of the differences on top
# sns.stripplot(data=pd.DataFrame({'first_half':first_half_pos - first_half_neg}), ax=ax[0], alpha=0.01, color='k')
# ax[0].hlines((first_half_pos - first_half_neg).mean(), -0.5, 0.5, color='k', linestyle='--')

# sns.stripplot(data=pd.DataFrame({'second_half':second_half_pos - second_half_neg}), ax=ax[1], alpha =0.01, color='k')
# ax[1].hlines((second_half_pos - second_half_neg).mean(), -0.5, 0.5, color='k', linestyle='--')

fig.tight_layout()
sns.despine()
ax[0].set(ylabel=f'{metric} (z)')

plt.savefig(f'{plot_dir}/{metric}_HPC_PFC_{band1}_{event}_half_split_rpe_tercile.pdf', dpi=300)



In [ ]:
second_half_neg.mean()

In [ ]:
source_region = 'HPC'
event = 'cue_on'
band1= 'theta'
band2= 'theta'
metric = 'psi'

savepath = f'/Volumes/T7 Shield/scratch/MemoryBandit/{metric}'

for target_region in ['dlPFC']:
    conn_group_data_pos = np.load(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_rew_index.npy')
    conn_group_data_neg = np.load(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_nrew_index.npy')
    # conn_group_data_pos = np.load(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_hit_index.npy')
    # conn_group_data_neg = np.load(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_miss_index.npy')
    # conn_group_data_pos2 = np.load(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_cr_index.npy')
    # conn_group_data_neg2 = np.load(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_fa_index.npy')



    fig, ax = plt.subplots(1, 1, figsize=(5, 3), dpi=300)

    # Let's look at l vs r for HPC-OFC
    meta_data = pd.read_csv(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_rew_index_df')
    meta_data.hemi = meta_data.hemi.str.lower()
    hemi_index = meta_data[meta_data.hemi.isin(['l', 'r'])].index.values
    print(len(hemi_index))

    good_data = conn_group_data_pos
    bad_data = conn_group_data_neg

    # good_data = np.concatenate([conn_group_data_pos[hemi_index, :], conn_group_data_pos2[hemi_index, :]])
    # bad_data = np.concatenate([conn_group_data_neg[hemi_index, :], conn_group_data_neg2[hemi_index, :]])

    T_obs, clusters, cluster_p_values, H0 = mne.stats.permutation_cluster_test([good_data, bad_data], 
                                                                                     n_permutations=1000, 
                                                                                     out_type='mask', seed=2)
    # T_obs, clusters, cluster_p_values, H0 = mne.stats.permutation_cluster_1samp_test(conn_group_data_neg, 
    #                                                                                  n_permutations=1000,
    #                                                                                  out_type='mask')
    print(cluster_p_values)
    # stderr = np.nanstd(conn_group_data_pos[hemi_index, :], axis=0) / np.sqrt(conn_group_data_pos[hemi_index, :].shape[0])
    # mean_data = np.nanmean(conn_group_data_pos[hemi_index, :], axis=0)

    stderr = np.nanstd(good_data, axis=0) / np.sqrt(good_data.shape[0])
    mean_data = np.nanmean(good_data, axis=0)

    if event == 'cue_on':
            times = np.linspace(0, 1000, conn_group_data_pos[hemi_index, :].shape[1])
    else:
        times = np.linspace(0, 1500, conn_group_data_pos[hemi_index, :].shape[1])
    # freqs = np.linspace(0, 500, conn_group_data_pos[:, :].shape[1])

    ax.plot(times, mean_data, color='green')
    ax.fill_between(times, mean_data - stderr, mean_data + stderr, color='green', alpha=0.3)

    stderr = np.nanstd(bad_data, axis=0) / np.sqrt(good_data.shape[0])
    mean_data = np.nanmean(bad_data, axis=0)

    ax.plot(times, mean_data, color='red')
    ax.fill_between(times, mean_data - stderr, mean_data + stderr, color='red', alpha=0.3)


    # stderr = np.nanstd(conn_group_data_pos2[hemi_index, :], axis=0) / np.sqrt(conn_group_data_pos2[hemi_index, :].shape[0])
    # mean_data = np.nanmean(conn_group_data_pos2[hemi_index, :], axis=0)


    # ax.plot(times, mean_data, color='lightgreen',)
    # ax.fill_between(times, mean_data - stderr, mean_data + stderr, color='lightgreen', alpha=0.3)

    # stderr = np.nanstd(conn_group_data_neg2[hemi_index, :], axis=0) / np.sqrt(conn_group_data_neg2[hemi_index, :].shape[0])
    # mean_data = np.nanmean(conn_group_data_neg2[hemi_index, :], axis=0)


    # ax.plot(times, mean_data, color='orange')
    # ax.fill_between(times, mean_data - stderr, mean_data + stderr, color='orange', alpha=0.3)

    # ax.hlines(0, freqs[0], freqs[-1], 'k')

    for i_c, c in enumerate(clusters):
        c = c[0]
        if (cluster_p_values[i_c] <= 0.05) & ((times[c.stop - 1] - times[c.start]) > 166.6):
            if target_region=='HPC':
                h = ax.axvspan(times[c.start], times[c.stop - 1], color="purple", alpha=0.3)
            elif target_region=='AMY':
                h = ax.axvspan(times[c.start], times[c.stop - 1], color="yellow", alpha=0.3)
            else: 
                h = ax.axvspan(times[c.start], times[c.stop - 1], color="k", alpha=0.3)
#         else:
#             ax.axvspan(times[c.start], times[c.stop - 1], color=(0.3, 0.3, 0.3), alpha=0.3)


    sns.despine()
    ax.set(xlabel='time (ms)', ylabel=f'{metric} (z)', ylim=[-0.75, 0.75])
    plt.savefig(f'{plot_dir}/{metric}_{source_region}_{target_region}_{band1}_{event}_rew.pdf', dpi=300)


In [ ]:
source_region = 'HPC'
event = 'cue_on'
band1= 'theta'
band2= 'theta'
metric = 'psi'

savepath = f'/Volumes/T7 Shield/scratch/MemoryBandit/{metric}'

for target_region in ['dlPFC']:
    conn_group_data_pos = np.load(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_rew_index.npy')
    conn_group_data_neg = np.load(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_nrew_index.npy')
    # conn_group_data_pos = np.load(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_hit_index.npy')
    # conn_group_data_neg = np.load(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_miss_index.npy')
    # conn_group_data_pos2 = np.load(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_cr_index.npy')
    # conn_group_data_neg2 = np.load(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_fa_index.npy')



    fig, ax = plt.subplots(1, 1, figsize=(5, 3), dpi=300)

    # Let's look at l vs r for HPC-OFC
    meta_data = pd.read_csv(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_rew_index_df')
    meta_data.hemi = meta_data.hemi.str.lower()
    hemi_index = meta_data[meta_data.hemi.isin(['l', 'r'])].index.values
    print(len(hemi_index))

    good_data = conn_group_data_pos
    bad_data = conn_group_data_neg

    # good_data = np.concatenate([conn_group_data_pos[hemi_index, :], conn_group_data_pos2[hemi_index, :]])
    # bad_data = np.concatenate([conn_group_data_neg[hemi_index, :], conn_group_data_neg2[hemi_index, :]])

    T_obs, clusters, cluster_p_values, H0 = mne.stats.permutation_cluster_test([good_data, bad_data], 
                                                                                     n_permutations=1000, 
                                                                                     out_type='mask', seed=2)
    # T_obs, clusters, cluster_p_values, H0 = mne.stats.permutation_cluster_1samp_test(conn_group_data_neg, 
    #                                                                                  n_permutations=1000,
    #                                                                                  out_type='mask')
    print(cluster_p_values)
    # stderr = np.nanstd(conn_group_data_pos[hemi_index, :], axis=0) / np.sqrt(conn_group_data_pos[hemi_index, :].shape[0])
    # mean_data = np.nanmean(conn_group_data_pos[hemi_index, :], axis=0)

    stderr = np.nanstd(good_data, axis=0) / np.sqrt(good_data.shape[0])
    mean_data = np.nanmean(good_data, axis=0)

    if event == 'cue_on':
            times = np.linspace(0, 1000, conn_group_data_pos[hemi_index, :].shape[1])
    else:
        times = np.linspace(0, 1500, conn_group_data_pos[hemi_index, :].shape[1])
    # freqs = np.linspace(0, 500, conn_group_data_pos[:, :].shape[1])

    ax.plot(times, mean_data, color='green')
    ax.fill_between(times, mean_data - stderr, mean_data + stderr, color='green', alpha=0.3)

    stderr = np.nanstd(bad_data, axis=0) / np.sqrt(good_data.shape[0])
    mean_data = np.nanmean(bad_data, axis=0)

    ax.plot(times, mean_data, color='red')
    ax.fill_between(times, mean_data - stderr, mean_data + stderr, color='red', alpha=0.3)


    # stderr = np.nanstd(conn_group_data_pos2[hemi_index, :], axis=0) / np.sqrt(conn_group_data_pos2[hemi_index, :].shape[0])
    # mean_data = np.nanmean(conn_group_data_pos2[hemi_index, :], axis=0)


    # ax.plot(times, mean_data, color='lightgreen',)
    # ax.fill_between(times, mean_data - stderr, mean_data + stderr, color='lightgreen', alpha=0.3)

    # stderr = np.nanstd(conn_group_data_neg2[hemi_index, :], axis=0) / np.sqrt(conn_group_data_neg2[hemi_index, :].shape[0])
    # mean_data = np.nanmean(conn_group_data_neg2[hemi_index, :], axis=0)


    # ax.plot(times, mean_data, color='orange')
    # ax.fill_between(times, mean_data - stderr, mean_data + stderr, color='orange', alpha=0.3)

    # ax.hlines(0, freqs[0], freqs[-1], 'k')

    for i_c, c in enumerate(clusters):
        c = c[0]
        if (cluster_p_values[i_c] <= 0.05) & ((times[c.stop - 1] - times[c.start]) > 166.6):
            if target_region=='HPC':
                h = ax.axvspan(times[c.start], times[c.stop - 1], color="purple", alpha=0.3)
            elif target_region=='AMY':
                h = ax.axvspan(times[c.start], times[c.stop - 1], color="yellow", alpha=0.3)
            else: 
                h = ax.axvspan(times[c.start], times[c.stop - 1], color="k", alpha=0.3)
#         else:
#             ax.axvspan(times[c.start], times[c.stop - 1], color=(0.3, 0.3, 0.3), alpha=0.3)


    sns.despine()
    ax.set(xlabel='time (ms)', ylabel=f'{metric} (z)', ylim=[-0.75, 0.75])
    plt.savefig(f'{plot_dir}/{metric}_{source_region}_{target_region}_{band1}_{event}_rew.pdf', dpi=300)


In [ ]:
source_region = 'HPC'
event = 'recog_time'
band1= 'theta'
band2= 'theta'
metric = 'coh'

savepath = f'/Volumes/T7 Shield/scratch/MemoryBandit/{metric}'

for target_region in ['dmPFC']:
    conn_group_data_pos = np.load(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_hit_index.npy')
    conn_group_data_neg = np.load(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_miss_index.npy')
    conn_group_data_pos2 = np.load(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_cr_index.npy')
    conn_group_data_neg2 = np.load(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_fa_index.npy')


    fig, ax = plt.subplots(1, 1, figsize=(5, 3), dpi=300)

    # Let's look at l vs r for HPC-OFC
    meta_data = pd.read_csv(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_miss_index_df')
    meta_data.hemi = meta_data.hemi.str.lower()
    hemi_index = meta_data[meta_data.hemi.isin(['l', 'r'])].index.values
    print(len(hemi_index))


    T_obs, clusters, cluster_p_values, H0 = mne.stats.permutation_cluster_test([conn_group_data_pos[hemi_index, :], conn_group_data_pos2[hemi_index, :]], 
                                                                                     n_permutations=1000, 
                                                                                     out_type='mask', seed=2)

    print(cluster_p_values)
    stderr = np.nanstd(conn_group_data_pos[hemi_index, :], axis=0) / np.sqrt(conn_group_data_pos[hemi_index, :].shape[0])
    mean_data = np.nanmean(conn_group_data_pos[hemi_index, :], axis=0)

    if event == 'cue_on':
            times = np.linspace(0, 1000, conn_group_data_pos[hemi_index, :].shape[1])
    else:
        times = np.linspace(0, 1500, conn_group_data_pos[hemi_index, :].shape[1])
    # freqs = np.linspace(0, 500, conn_group_data_pos[:, :].shape[1])

    ax.plot(times, mean_data, color='green')
    ax.fill_between(times, mean_data - stderr, mean_data + stderr, color='green', alpha=0.3)

    stderr = np.nanstd(conn_group_data_pos2[hemi_index, :], axis=0) / np.sqrt(conn_group_data_pos2[hemi_index, :].shape[0])
    mean_data = np.nanmean(conn_group_data_pos2[hemi_index, :], axis=0)

    ax.plot(times, mean_data, color='lightgreen')
    ax.fill_between(times, mean_data - stderr, mean_data + stderr, color='lightgreen', alpha=0.3)

    for i_c, c in enumerate(clusters):
        c = c[0]
        if (cluster_p_values[i_c] <= 0.05) & ((times[c.stop - 1] - times[c.start]) > 166.6):
            if target_region=='HPC':
                h = ax.axvspan(times[c.start], times[c.stop - 1], color="purple", alpha=0.3)
            elif target_region=='AMY':
                h = ax.axvspan(times[c.start], times[c.stop - 1], color="yellow", alpha=0.3)
            else: 
                h = ax.axvspan(times[c.start], times[c.stop - 1], color="k", alpha=0.3)
#         else:
#             ax.axvspan(times[c.start], times[c.stop - 1], color=(0.3, 0.3, 0.3), alpha=0.3)


    sns.despine()
    ax.set(xlabel='time (ms)', ylabel=f'{metric} (z)', ylim=[-0.75, 0.75])
    plt.savefig(f'{plot_dir}/{metric}_{source_region}_{target_region}_{band1}_{event}_hitvcr_Z.pdf', dpi=300)


In [ ]:
T_obs.max()

In [ ]:
good_data.shape

In [ ]:
good_data.shape

In [ ]:
meta_data

In [ ]:
conn_group_data_pos.shape

In [ ]:
hemi_index

In [ ]:
times[c.stop - 1]

In [ ]:
conn_group_data_pos[conn_group_metadata[conn_group_metadata.hemi=='l'].index.values, :]

In [ ]:
conn_group_metadata[conn_group_metadata.hemi=='l'].index.values

In [ ]:
f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_hit_index.npy'


In [ ]:
'/Volumes/T7 Shield/scratch/MemoryBandit/psi/HPC_OFC_psi_theta_theta_feedback_start_hit_index.npy'

In [ ]:
# grouped across subjects, pairwise granger is utterly unimpressive during both feedback and cue

In [ ]:
meta_data = pd.read_csv(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_miss_Z_df')

In [ ]:
meta_data['conn'] = np.nanmean(conn_group_data_neg, axis=1)

In [ ]:
meta_data

In [ ]:
# drop one subject

source_region = 'HPC'
target_region = 'OFC'
band1= 'theta'
band2= 'theta'
event = 'feedback_start'
metric = 'coh'

fig, ax = plt.subplots(1, 1, figsize=(5, 3), dpi=300)
for target_region in ['OFC']:
    conn_group_data_pos = np.load(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_hit_Z.npy')
    conn_group_data_neg = np.load(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_miss_Z.npy')

    for subj in meta_data.participant.unique():
        fig, ax = plt.subplots(1, 1, figsize=(5, 3), dpi=300)

        subj_index = meta_data[meta_data.participant==subj].index.values

        T_obs, clusters, cluster_p_values, H0 = mne.stats.permutation_cluster_test([np.delete(conn_group_data_pos, subj_index, axis=0),
                                                                                    np.delete(conn_group_data_neg, subj_index, axis=0)], 
                                                                                         n_permutations=1000,
                                                                                         out_type='mask')
        print(cluster_p_values)
        stderr = np.nanstd(np.delete(conn_group_data_pos, subj_index, axis=0), axis=0) / np.sqrt(np.delete(conn_group_data_pos, subj_index, axis=0).shape[0])
        mean_data = np.nanmean(np.delete(conn_group_data_pos, subj_index, axis=0), axis=0)

        freqs = np.linspace(0, 1500, conn_group_data_pos[:, :].shape[1])

        ax.plot(freqs, mean_data)
        ax.fill_between(freqs, mean_data - stderr, mean_data + stderr, color='green', alpha=0.3)

        stderr = np.nanstd(np.delete(conn_group_data_neg, subj_index, axis=0), axis=0) / np.sqrt(np.delete(conn_group_data_neg, subj_index, axis=0).shape[0])
        mean_data = np.nanmean(np.delete(conn_group_data_neg, subj_index, axis=0), axis=0)


        ax.plot(freqs, mean_data)
        ax.fill_between(freqs, mean_data - stderr, mean_data + stderr, color='red', alpha=0.3)

        ax.hlines(0, 2, 8, 'k')

        for i_c, c in enumerate(clusters):
            c = c[0]
            if cluster_p_values[i_c] <= 0.05:
                if target_region=='HPC':
                    h = ax.axvspan(freqs[c.start], freqs[c.stop - 1], color="purple", alpha=0.3)
                elif target_region=='AMY':
                    h = ax.axvspan(freqs[c.start], freqs[c.stop - 1], color="yellow", alpha=0.3)
                else: 
                    h = ax.axvspan(freqs[c.start], freqs[c.stop - 1], color="k", alpha=0.3)
    #         else:
    #             ax.axvspan(times[c.start], times[c.stop - 1], color=(0.3, 0.3, 0.3), alpha=0.3)


        sns.despine()
        ax.set(xlabel='freqs (Hz)', ylabel='theta PSI')
        plt.savefig(f'{plot_dir}/{metric}_{source_region}_{target_region}_{band1}_memory_Z.pdf', dpi=300)


In [ ]:
conn_group_data_pos

In [ ]:
np.delete(conn_group_data_pos, subj_index, axis=0).shape


In [ ]:
subj_index

In [ ]:
ttest_rel(conn_group_data_pos.mean(axis=1), 
          conn_group_data_neg.mean(axis=1))

In [ ]:
conn_group_data_neg.mean()

In [ ]:
clusters

In [ ]:
cluster_p_values

In [ ]:
conn_group_metadata

In [ ]:
# for target_region in ['HPC', 'OFC', 'dmPFC']:
#     conn_group_data = np.load(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}.npy')
#     plt.plot(np.nanmean(conn_group_data, axis=0))
    

In [ ]:
epochs_reref._data.shape

In [ ]:
seed_to_target.shape

In [ ]:
conn_group_data.shape

In [ ]:
conn_group_metadata.shape

In [ ]:
psi = np.squeeze(test_data.get_data()[:, 0])[:, buf_rs:-buf_rs]

In [ ]:
plt.plot(np.nanmean(psi, axis=0))

In [ ]:
pwise = oscillation_utils.compute_connectivity(epochs_reref.copy(), 
                                               band = freq_dict[band1], 
                                               metric = metric, 
                                               indices = seed_to_target, 
                                               freqs = freqs, 
                                               n_cycles = n_cycles,
                                               buf_ms = buf_ms, 
                                               n_surr=0,
                                               avg_over_dim='epochs',
                                               band1 = freq_dict[band2],
                                               parallelize=True)

In [ ]:
seed_to_target

In [ ]:
plt.plot(np.nanmean(pwise, axis=0))

## Undirected connectivity (e.g. coh) - computed over timepoints

In [ ]:
import mne

In [ ]:
analysis_evs = ['feedback_start']

metric = 'granger'
freqs = np.logspace(*np.log10([2, 200]), num=30)
n_cycles = np.floor(np.logspace(*np.log10([3, 10]), num=30))

freq_dict = {'theta':[4, 8], 
            'alpha':[8, 13],
            'beta': [13, 30], 
            'hfa': [70, 200]}

# freq_dict = {'beta':[4, 8]}

band1 = 'beta'
band2 = 'beta'
day = 'Day1'

buf_ms = 1000
n_surr = 0

savepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/Connectivity/scratch/{metric}'
if not os.path.exists(savepath):
    os.makedirs(savepath)

## First, let's compute it for the whole session

In [ ]:
pairs = (['HPC', 'OFC'], ['HPC', 'dmPFC'], ['HPC', 'ACC'], ['HPC', 'dlPFC'],)
# time_window = [0, 0.75]
time_window = [0.75, 1.5]

for pair in pairs:
    source_region = pair[0]
    target_region = pair[1]
    
    # initialize the storage list 
    conn_group_data = []

    # iterate through subjects
    for subj_id in tqdm(subj_ids, desc=f'{source_region} to {target_region} connectivity using {metric}'):
        # display(f'Computing granger causality for {pair} in {subj_id}')
        for event in analysis_evs:
            loadpath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}'
            epochs_reref = mne.read_epochs(f'{loadpath}/neural/{day}/{event}-epo.fif', preload=True) 
                     
            # Let's get rid of the trials we have pre-emptively dropped 
            all_trials = np.arange(epochs_reref._data.shape[0])
            valid_trials = learn_df[learn_df.participant==subj_id].trials.values
            valid_trials_index = valid_trials -1 
            bad_trial_index = list( set(valid_trials_index) ^ set(all_trials))
            epochs_reref.drop(bad_trial_index, reason='Bad Behavior')

            # Get electrode df 
            elec_df = pd.read_csv(f'{loadpath}/Day1_reref_elec_df')

            seed_target_df = oscillation_utils.make_seed_target_df(elec_df, epochs_reref, source_region, target_region)
            
            # For within-region connectivity, let's drop elements that are the same
            
            if len(seed_target_df) == 0:
                continue
                
            # for cond in conditions: 
            for hemi in ['l', 'r']:
                # first determine if ipsilateral connectivity is even possible; if not, move on. 
                # note: edit this if you want to compute contralateral connectivity 
                if hemi not in seed_target_df.index.tolist():
                    continue
                else:
                    seed_to_target = seed_target_indices(
                        seed_target_df['seed'][hemi],
                        seed_target_df['target'][hemi])
                
                if metric == 'amp':
                    pwise = oscillation_utils.compute_connectivity(epochs_reref.copy(), 
                                               band = freq_dict[band1], 
                                               metric = metric, 
                                               indices = seed_to_target, 
                                               freqs = None, 
                                               n_cycles = None,
                                               buf_ms = buf_ms, 
                                               n_surr=n_surr,
                                               avg_over_dim='time',
                                               band1 = freq_dict[band2],
                                               time_window = time_window)

                if metric == 'gcmi':
                    pwise = oscillation_utils.compute_connectivity(epochs_reref.copy(), 
                                               band = freq_dict[band1], 
                                               metric = metric, 
                                               indices = seed_to_target, 
                                               freqs = freqs, 
                                               n_cycles = n_cycles,
                                               buf_ms = buf_ms, 
                                               n_surr=n_surr,
                                               avg_over_dim='time',
                                               band1 = freq_dict[band2],
                                               time_window = time_window)
                    
                elif metric == 'granger': 
                    pwise = oscillation_utils.compute_connectivity(epochs_reref.copy(), 
                                               band = freq_dict[band1], 
                                               metric = metric, 
                                               indices = seed_to_target, 
                                               freqs = freqs, 
                                               n_cycles = n_cycles,
                                               buf_ms = buf_ms, 
                                               n_surr=n_surr,
                                               avg_over_dim='time',
                                               band1 = freq_dict[band2],
                                               parallelize=True,
                                               gc_n_lags=5,
                                               time_window = time_window)
                    
                else:
                    pwise = oscillation_utils.compute_connectivity(epochs_reref.copy(), 
                                               band = freq_dict[band1], 
                                               metric = metric, 
                                               indices = seed_to_target, 
                                               freqs = freqs, 
                                               n_cycles = n_cycles,
                                               buf_ms = buf_ms, 
                                               n_surr=n_surr,
                                               avg_over_dim='time',
                                               surr_method = 'swap_time_blocks',
                                               band1 = freq_dict[band2],
                                               parallelize=True, 
                                               time_window = time_window)

                pwise_dfs = []

                if metric in ['gc', 'granger', 'cacoh']:
                    # no pairs here: computed over whole multivariate state space 
                    npairs=1
                else:
                    npairs = len(seed_to_target[0])
                    
                for pair in range(npairs): 
                    if epochs_reref.ch_names[seed_to_target[0][pair]] == epochs_reref.ch_names[seed_to_target[1][pair]]:
                        # skip any same-channel connectivity
                        continue
                    pwise_df = pd.DataFrame(columns=['participant', 'roi1', 'roi2', 'hemi', 'pair_label', 'metric', 'event', 'conn'])
                    pwise_df['conn'] = pwise[:, pair] 
                    pwise_df['participant'] = subj_id
                    pwise_df['age'] = subj_df[subj_df.MSSMCode==subj_id].Age.values[0]
                    pwise_df['sex'] = subj_df[subj_df.MSSMCode==subj_id].Sex.str.strip().values[0]
                    pwise_df['roi1'] = source_region
                    pwise_df['roi2'] = target_region
                    pwise_df['hemi'] = hemi 
                    pwise_df['metric'] = metric
                    pwise_df['event'] = event
                    pwise_df['band0'] = band1
                    pwise_df['band1'] =  band2
                    if time_window == [0, 0.75]:
                        pwise_df['time_window'] = 'first_half'
                    elif time_window == [0.75, 1.5]:
                        pwise_df['time_window'] = 'second_half'
                    # pwise_df['trials'] = valid_trials
                    
                    if metric=='granger':
                        pwise_df['pair_label'] = f'{source_region}:{target_region}'
                    else:
                        pwise_df['pair_label'] = f'{epochs_reref.ch_names[seed_to_target[0][pair]]}:{epochs_reref.ch_names[seed_to_target[1][pair]]}'
                    # pwise_df['trials'] = np.arange(1, pwise.shape[0]+1)
                    if event in ['gamble_start', 'feedback_start', 'baseline_start']:
                        pwise_df['trials'] = valid_trials
                        pwise_df = pwise_df.merge(learn_df[(learn_df.participant==subj_id)], on=['participant', 'trials'])
                        # pwise_df['trials'] = np.arange(1, learn_df[learn_df.participant==subj_id].shape[0]+1)
                        # pwise_df = pwise_df.merge(learn_df[(learn_df.participant==subj_id)], on=['participant', 'trials'])
                    elif event in ['cue_on', 'recog_time']:
                        combined_df = combined_df[combined_df.recog_rt>1] 
                        pwise_df['trials_mem'] =  np.arange(1, pwise_df.shape[0]+1)
                        # combined_df[(combined_df.participant==subj_id) & (combined_df.condition=='Day1')].sort_values(by='trials_mem').reset_index(drop=True).trials_mem.values
                        pwise_df = pwise_df.merge(combined_df[(combined_df.participant==subj_id) & (combined_df.condition=='Day1')].sort_values(by='trials_mem').reset_index(drop=True), 
                                                  on=['participant', 'trials_mem'])

                    # pwise_df['zrpe'] = (pwise_df.rpe - np.nanmean(pwise_df.rpe)) / (2*np.nanstd(pwise_df.rpe))
                    # pwise_df['zpm'] = (pwise_df.DPRIME - np.nanmean(pwise_df.DPRIME)) / (2*np.nanstd(pwise_df.DPRIME))

                    # aggregate
                    pwise_dfs.append(pwise_df)
                pwise_dfs = pd.concat(pwise_dfs)
                conn_group_data.append(pwise_dfs)

    all_pairs_df = pd.concat(conn_group_data)
    if time_window == [0, 0.75]:
        all_pairs_df.to_csv(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_first_half_df', index=False)
    elif time_window == [0.75, 1.5]:
        all_pairs_df.to_csv(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_second_half_df', index=False)



In [ ]:
all_merged_dfs = []
for target_region in ['OFC', 'dmPFC', 'ACC', 'dlPFC']:
    first_half_df = pd.read_csv(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_first_half_df')
    first_half_df['time_window'] = 'first_half'
    second_half_df = pd.read_csv(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_second_half_df')
    second_half_df['time_window'] = 'second_half'

    merged_df = first_half_df.merge(
        second_half_df, 
        on=[col for col in first_half_df.columns if col not in ['time_window', 'conn']],
        suffixes=('_first_half', '_second_half')
    )

    all_merged_dfs.append(merged_df)

all_merged_dfs = pd.concat(all_merged_dfs)


In [ ]:
sns.barplot(data=all_merged_dfs, x='rpe_cat', y='conn_second_half', hue='roi2', errorbar='se')

In [ ]:
all_merged_dfs['unique_pair_label'] = all_merged_dfs.participant + '_' + all_merged_dfs.pair_label

In [ ]:
from scipy.stats import ttest_rel

# Run paired t-tests for each condition (rpe_cat) and pair_label
results = []

for (rpe_cat, pair_label), group in all_merged_dfs.groupby(['rpe_cat', 'unique_pair_label']):
    # Paired t-test between first half and second half connectivity
    t_stat, p_val = ttest_rel(group['conn_first_half'].values, group['conn_second_half'].values)
    
    results.append({
        'participant': group['participant'].iloc[0],
        'roi1': group['roi1'].iloc[0],
        'roi2': group['roi2'].iloc[0],
        'hemi': group['hemi'].iloc[0],
        'metric': group['metric'].iloc[0],
        'event': group['event'].iloc[0],
        'band0': group['band0'].iloc[0],
        'band1': group['band1'].iloc[0],
        'rpe_cat': rpe_cat,
        'pair_label': pair_label,
        't_stat': t_stat,
        'p_val': p_val,
        'mean_first_half': group['conn_first_half'].mean(),
        'mean_second_half': group['conn_second_half'].mean(),
        'n': len(group)
    })

results_df = pd.DataFrame(results)

In [ ]:
sns.barplot(data=results_df, x='rpe_cat', y='t_stat', hue='roi2', errorbar='se')

In [ ]:
import statsmodels.formula.api as smf
smf.mixedlm("t_stat ~ 1 + C(rpe_cat, Treatment('neg'))", 
data=results_df[results_df.roi2=='ACC'], groups='participant').fit().summary()

In [ ]:
ttest_rel(group['conn_first_half'].values, 
group['conn_second_half'].values)

In [ ]:
t_stat

In [ ]:
group['conn_first_half'].values

In [ ]:
results_df

In [ ]:
source_region = 'AMY'
band1 = 'hfa'
band2 = 'hfa'
event = 'feedback_start'
metric = 'amp'
savepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/Connectivity/scratch/{metric}'


df = [] 
for target_roi in ['OFC', 'dmPFC', 'ACC', 'dlPFC']:
    df.append(pd.read_csv(f'{savepath}/{source_region}_{target_roi}_{metric}_{band1}_{band2}_{event}_df_surr'))
    # df.append(pd.read_csv(f'{savepath}/{source_region}_{target_roi}_{metric}_{band1}_{band2}_{event}_df_raw'))
df_encoding = pd.concat(df)
df_encoding[['participant', 'trials', 'pair_label', 'roi2', 'conn', 'hemi']].rename(columns={'trials':'trials_gamble'})
df_encoding = df_encoding[['participant', 'trials', 'pair_label', 'roi2', 'conn', 'hemi']].rename(columns={'trials':'trials_gamble'}).merge(combined_df, on=['participant', 'trials_gamble'])
df_encoding['unique_id'] = df_encoding.participant + '_' + df_encoding.pair_label.astype(str)

In [ ]:
df_encoding = df_encoding[df_encoding.condition=='Day1']


In [ ]:
# Basic mixed-effects model: 
import statsmodels.formula.api as smf
print(smf.mixedlm("conn ~ 1 + C(hemi)*C(roi2, Treatment('dmPFC')) + zrpe*phit", 
                  df_encoding, 
                  groups='participant').fit().summary())

In [ ]:
sns.lmplot(data=df_encoding, x='zrpe', y='conn', hue='phit', row='roi2', scatter=False, ci=95)

In [ ]:
source_region = 'HPC'
target_region = 'dlPFC'
band1 = 'theta'
band2 = 'theta'
event = 'cue_on'
metric = 'coh'
savepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/Connectivity/scratch/{metric}'


df = [] 
for target_roi in ['OFC', 'dmPFC', 'ACC', 'dlPFC']:
    df.append(pd.read_csv(f'{savepath}/{source_region}_{target_roi}_{metric}_{band1}_{band2}_{event}_df_raw'))
    # all_pairs_df = pd.read_csv(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_df_raw')
df_cue = pd.concat(df)

In [ ]:
df_cue['goodvbadmemory'] = df_cue.memory_outcome.isin(['hits', 'correct_rejections']).astype(int)

In [ ]:
for roi in ['OFC', 'dmPFC', 'ACC', 'dlPFC']:
    print(smf.mixedlm("conn ~ 1 + C(recog_choice) + recog_rt_log*C(goodvbadmemory)", 
                df_cue[df_cue.roi2==roi], 
                groups='participant').fit().summary())

In [ ]:
# grouped_df_encoding = df_encoding.groupby(['participant', 'pair_label', 'roi2', 'reward']).mean(numeric_only=True).reset_index()
# grouped_df_cue = df_cue.groupby(['participant', 'pair_label', 'roi2', 'memory_outcome']).mean(numeric_only=True).reset_index()

In [ ]:
merged_df = pd.merge(df_encoding, df_cue, on=['participant', 'pair_label', 'roi2', 'img_path'], suffixes=('_encoding', '_cue'))

In [ ]:
from scipy.stats import pearsonr
for roi in ['OFC', 'dmPFC', 'ACC', 'dlPFC']:
    for memory in ['hits', 'misses']:
        print(roi, memory)
        print(pearsonr(merged_df[(merged_df.roi2==roi) & (merged_df.memory_outcome==memory)].conn_encoding, 
                       merged_df[(merged_df.roi2==roi) & (merged_df.memory_outcome==memory)].conn_cue))


In [ ]:
sns.lmplot(data=merged_df, x='conn_encoding', y='conn_cue', col='roi2', scatter=False)

In [ ]:
f, ax = plt.subplots(1, 1, figsize=(3, 4), dpi=300)

# make a stripplot to match
sns.stripplot(data=df.groupby(['pair_label', 'roi2', 'goodvbadmemory']).mean(numeric_only=True).reset_index(),
              y='conn', 
              x='roi2',
              ax=ax, 
              hue='goodvbadmemory',
              dodge=True,
              palette=['red', 'green'],
              order=['dmPFC', 'ACC', 'dlPFC', 'OFC'], 
              alpha=0.05, legend=None)

# plot a horizontal line at the mean value for each roi2
for i, roi in enumerate(['dmPFC', 'ACC', 'dlPFC', 'OFC']):
    ax.hlines(df[(df.roi2==roi) & (df.goodvbadmemory==0)].conn.mean(), i-0.4, i, color='red')
    ax.hlines(df[(df.roi2==roi) & (df.goodvbadmemory==1)].conn.mean(), i, i+0.4, color='green')

ax.set(ylim=[-0.5, 0.7])
sns.despine()

# plt.savefig(f'{savepath}/{source_region}_{metric}_{band1}_{band2}_{event}_grouped_bars.pdf', dpi=300)



In [ ]:
smf.mixedlm("conn ~ hemi + recog_rt_log + C(goodvbadmemory)", 
            data=df[df.roi2=='OFC'], groups='pair_label').fit().summary()

In [ ]:
smf.mixedlm("conn ~ hemi + recog_rt_log + C(memory_outcome, Treatment('hits'))", 
            data=df[df.roi2=='dlPFC'], groups='pair_label').fit().summary()

In [ ]:
f, ax = plt.subplots(1, 1, figsize=(3, 4), dpi=300)

# make a stripplot to match
sns.stripplot(data=df[df.memory_outcome.isin(['hits','correct_rejections'])].groupby(['pair_label', 'roi2', 'memory_outcome']).mean(numeric_only=True).reset_index(),
              y='conn', 
              x='roi2',
              ax=ax, 
              hue='memory_outcome',
              dodge=True,
              palette=['red', 'green'],
              order=['dmPFC', 'ACC', 'dlPFC', 'OFC'], 
              alpha=0.05, legend=None)

# plot a horizontal line at the mean value for each roi2
for i, roi in enumerate(['dmPFC', 'ACC', 'dlPFC', 'OFC']):
    ax.hlines(df[(df.roi2==roi) & (df.memory_outcome=='correct_rejections')].conn.mean(), i-0.4, i, color='red')
    ax.hlines(df[(df.roi2==roi) & (df.memory_outcome=='hits')].conn.mean(), i, i+0.4, color='green')

ax.set(ylim=[-0.5, 0.7])
sns.despine()


In [ ]:
df = df[['participant', 'trials', 'pair_label', 'roi2', 'conn', 'hemi', 'reward']].rename(columns={'trials':'trials_gamble'}).merge(combined_df, on=['participant', 'trials_gamble'])

# # Plot conn as a function of reward and hits on different axes
import statsmodels.formula.api as smf
smf.mixedlm("conn ~ 1 + rpe*hits", 
            data=df[df.roi2=='dlPFC'],
            groups='participant').fit().summary()


In [ ]:
f, ax = plt.subplots(1, 1, figsize=(3, 4), dpi=300)

# make a stripplot to match
sns.stripplot(data=df.groupby(['pair_label', 'roi2', 'reward']).mean(numeric_only=True).reset_index(),
              y='conn', 
              x='roi2',
              ax=ax, 
              hue='reward',
              dodge=True,
              palette=['gray', 'black'],
              order=['dmPFC', 'ACC', 'dlPFC', 'OFC'], 
              alpha=0.05, legend=None)

# plot a horizontal line at the mean value for each roi2
for i, roi in enumerate(['dmPFC', 'ACC', 'dlPFC', 'OFC']):
    ax.hlines(df[(df.roi2==roi) & (df.reward==0)].conn.mean(), i-0.4, i, color='gray')
    ax.hlines(df[(df.roi2==roi) & (df.reward==1)].conn.mean(), i, i+0.4, color='black')

ax.set(ylim=[-0.5, 0.7])
sns.despine()

# plt.savefig(f'{savepath}/{source_region}_{metric}_{band1}_{band2}_{event}_grouped_bars.pdf', dpi=300)



In [ ]:
f, ax = plt.subplots(1, 1, figsize=(3, 4), dpi=300)

df = df[['participant', 'trials', 'pair_label', 'roi2', 'conn', 'hemi']].rename(columns={'trials':'trials_gamble'}).merge(combined_df, on=['participant', 'trials_gamble'])

# make a stripplot to match
sns.stripplot(data=df.groupby(['pair_label', 'roi2', 'hits']).mean(numeric_only=True).reset_index(),
              y='conn', 
              x='roi2',
              ax=ax, 
              hue='hits',
              palette=['red', 'green'],
              dodge=True,
              order=['dmPFC', 'ACC', 'dlPFC', 'OFC'], 
              alpha=0.05,
              legend=None)

# plot a horizontal line at the mean value for each roi2
for i, roi in enumerate(['dmPFC', 'ACC', 'dlPFC', 'OFC']):
    ax.hlines(df[(df.roi2==roi) & (df.hits==0)].conn.mean(), i-0.4, i, color='red')
    ax.hlines(df[(df.roi2==roi) & (df.hits==1)].conn.mean(), i, i+0.4, color='green')

ax.set(ylim=[-0.5, 0.7])
sns.despine()

# plt.savefig(f'{savepath}/{source_region}_{metric}_{band1}_{band2}_{event}_grouped_bars.pdf', dpi=300)



In [ ]:
f, ax = plt.subplots(1, 1, figsize=(3, 4), dpi=300)

# make a stripplot to match
sns.stripplot(data=df.groupby(['pair_label', 'roi2']).mean(numeric_only=True).reset_index(),
              y='conn', 
              x='roi2',
              ax=ax, 
              color='k', 
              order=['dmPFC', 'ACC', 'dlPFC', 'OFC'], 
              alpha=0.05)

# plot a horizontal line at the mean value for each roi2
for i, roi in enumerate(['dmPFC', 'ACC', 'dlPFC', 'OFC']):
    ax.hlines(df[df.roi2==roi].conn.mean(), i-0.2, i+0.2, color='k')


ax.set(ylim=[-0.5, 0.7])
sns.despine()

plt.savefig(f'{savepath}/{source_region}_{metric}_{band1}_{band2}_{event}_grouped_bars.pdf', dpi=300)



In [ ]:
import statsmodels.formula.api as smf
smf.mixedlm("conn ~ 1 + C(roi2, Treatment('dmPFC'))", 
            data=df,
            groups='participant').fit().summary()

In [ ]:

df = df[['participant', 'trials', 'pair_label', 'roi2', 'conn', 'hemi']].rename(columns={'trials':'trials_gamble'}).merge(combined_df, on=['participant', 'trials_gamble'])

In [ ]:
f, ax = plt.subplots(1, 1, figsize=(3, 4), dpi=300)

# make a stripplot to match
sns.stripplot(data=df.groupby(['pair_label', 'roi2']).mean(numeric_only=True).reset_index(),
              y='conn', 
              x='roi2',
              ax=ax, 
              color='k', 
              order=['dmPFC', 'ACC', 'dlPFC', 'OFC'], 
              alpha=0.05)

# plot a horizontal line at the mean value for each roi2
for i, roi in enumerate(['dmPFC', 'ACC', 'dlPFC', 'OFC']):
    ax.hlines(df[df.roi2==roi].conn.mean(), i-0.2, i+0.2, color='k')


sns.despine()

plt.savefig(f'{savepath}/{source_region}_{metric}_{band1}_{band2}_{event}_grouped_bars.pdf', dpi=300)



In [ ]:
savepath

In [ ]:
smf.mixedlm("conn ~ 1 + C(roi2, Treatment('dmPFC'))", 
            data=df,
            groups='participant').fit().summary()


In [ ]:
f, ax = plt.subplots(1, 1, figsize=(10, 5), dpi=300)
sns.barplot(data=df, 
            x='memory_outcome', 
            y='conn', 
            hue='roi2',
            ax=ax,
            errorbar='se', hue_order=['dmPFC', 'ACC', 'dlPFC', 'OFC'])
# # plot stripplot on top
# sns.stripplot(data=df.groupby(['participant', 'roi2']).mean(numeric_only=True).reset_index(),
#               x='roi2', 
#               y='conn', 
#               ax=ax, 
#               color='k', 
#               order=['dmPFC', 'ACC', 'dlPFC', 'OFC'],
#               alpha=0.5)
# import statsmodels.formula.api as smf
# # make treatment variable dmPFC
# df['roi2'] = pd.Categorical(df['roi2'], categories=['dmPFC', 'ACC', 'dlPFC', 'OFC'], ordered=True)
# plt.savefig(f'{base_dir}/work/qasims01/MemoryBanditData/EMU/plots/{source_region}_{metric}_{band1}_{band2}_{event}_time_avg_conn_bar.pdf', dpi=300)

In [ ]:
smf.mixedlm("conn ~ 1 + C(memory_outcome, Treatment('hits'))", 
            data=df[df.roi2=='dmPFC'],
            re_formula="1 + C(memory_outcome, Treatment('hits'))",
            groups='pair_label').fit().summary()



So HPC-OFC is the most significant theta coherence connection... then, when we look at directional stuff we can look just at HPC-OFC PSI for directionality.

Regression strategy:

Shouldn't I run this on a single-electrode level rather than a mixed-effects?

## Encoding

In [ ]:
import statsmodels.formula.api as smf
pairs = (['HPC', 'OFC'], ['HPC', 'dmPFC'], ['HPC', 'ACC'], ['HPC', 'dlPFC'],)
band1 = 'beta'
band2 = 'beta'
event = 'feedback_start'
# pairs = ['AMY', 'HPC'], 
# pairs = (['HPC', 'OFC'], )

for pair in pairs:
    source_region = pair[0]
    target_region = pair[1]
    all_pairs_df = pd.read_csv(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_df_raw')
    # index unique pairs 
    all_pairs_df['unique_pair'] = all_pairs_df['participant'] + all_pairs_df['pair_label']

    # # Get the memory information: 
    memory_df = combined_df[(combined_df.condition=='Day1')].dropna(subset=['trials_gamble']).sort_values(by=['trials_gamble'])[['participant', 'trials_gamble', 'hits']]
    memory_df.rename(columns={'trials_gamble':'trials'}, inplace=True)
    memory_df['phit'] = np.where(memory_df.hits == True, 1, 0)
    all_pairs_df = all_pairs_df.merge(memory_df, on=['participant', 'trials'])

    all_pairs_df.dropna(subset=['phit'], inplace=True)
    all_pairs_df['zrpe'] = (all_pairs_df.rpe - np.nanmean(all_pairs_df.rpe)) / (np.nanstd(all_pairs_df.rpe))
    
    all_pairs_df['phit'] = all_pairs_df['phit'].astype(int)
    print(smf.mixedlm(formula='conn ~ 1 + phit*zrpe', data=all_pairs_df, groups=all_pairs_df['unique_pair']).fit().summary())
    # f, ax = plt.subplots(1, 1, figsize=(5, 3), dpi=300)
    # sns.regplot(x='alpha', y='conn', data=all_pairs_df, ax=ax)
    print(f'{source_region}-{target_region}')

In [ ]:
sns.lmplot(data=all_pairs_df, x='zrpe', y='conn', hue='phit', scatter=False, ci=95)

1/31/2025: No significant beta coherence as a function of hits:rpe

1/31/2025: Significant alpha coherence decrease between hippocampus and OFC as a function of hits:rpe


1/31/2025: Significant theta coherence increase between hippocampus and dlPFC as a function of hits:rpe
    - but this does not survive a single-electrode z-scoring strategy

## Retrieval

In [ ]:
import statsmodels.formula.api as smf
pairs = (['HPC', 'OFC'],) 
metric = 'coh'
band1 = 'beta'
band2 = 'beta'
event = 'cue_on'

for pair in pairs:
    source_region = pair[0]
    target_region = pair[1]
    all_pairs_df = pd.read_csv(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_df_raw')
    # index unique pairs 
    all_pairs_df['unique_pair'] = all_pairs_df['participant'] + all_pairs_df['pair_label']
    all_pairs_df['good_memory'] = all_pairs_df.memory_outcome.apply(lambda x: 'good' if x in ['hits', 'correct_rejections'] else 'bad')

    # # # Get the memory information: 
    # memory_df = combined_df[(combined_df.condition=='Day1')].dropna(subset=['trials_gamble']).sort_values(by=['trials_gamble'])[['participant', 'trials_gamble', 'hits']]
    # memory_df.rename(columns={'trials_gamble':'trials'}, inplace=True)
    # memory_df['phit'] = np.where(memory_df.hits == True, 1, 0)
    # all_pairs_df = all_pairs_df.merge(memory_df, on=['participant', 'trials'])

    # all_pairs_df.dropna(subset=['phit'], inplace=True)
    # all_pairs_df['zrpe'] = (all_pairs_df.rpe - np.nanmean(all_pairs_df.rpe)) / (np.nanstd(all_pairs_df.rpe))
    
    # all_pairs_df['phit'] = all_pairs_df['phit'].astype(int)
    print(smf.mixedlm(formula="conn ~ 1 + C(good_memory, Treatment(reference='bad'))", 
                      data=all_pairs_df, groups=all_pairs_df['unique_pair']).fit().summary())
    # f, ax = plt.subplots(1, 1, figsize=(5, 3), dpi=300)
    # sns.regplot(x='alpha', y='conn', data=all_pairs_df, ax=ax)
    print(f'{source_region}-{target_region}')

    

In [ ]:
sns.barplot(data=all_pairs_df, x='good_memory', y='conn', errorbar='se')

In [ ]:
all_dfs

In [ ]:
all_dfs

In [ ]:
all_dfs

In [ ]:
source_region = 'HPC'
target_region = 'OFC'
band1 = 'theta'
band2 = 'theta'
all_pairs_df = pd.read_csv(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_df_raw')
# index unique pairs 
all_pairs_df['unique_pair'] = all_pairs_df['participant'] + all_pairs_df['pair_label']
# add memory information 
all_pairs_df['hits'] = np.nan
for subj in all_pairs_df.participant.unique():
    for trial_mem in combined_df[(combined_df.participant==subj) & (combined_df.condition=='Day1')].dropna(subset=['trials_gamble']).sort_values(by='trials_mem').reset_index(drop=True).trials_mem.values:
        all_pairs_df.loc[(all_pairs_df.participant==subj) & (all_pairs_df.trials==trial_mem), 'hits'] = combined_df.loc[(combined_df.participant==subj_id) & (combined_df.trials_mem==trial_mem)].hits.values[0]

all_pairs_df.dropna(subset=['hits'], inplace=True)
all_pairs_df['zrpe'] = (all_pairs_df.rpe - np.nanmean(all_pairs_df.rpe)) / (np.nanstd(all_pairs_df.rpe))

all_pairs_df['hits'] = all_pairs_df['hits'].astype(int)

In [ ]:
sns.lmplot(data=all_pairs_df, x='zrpe', y='conn', hue='hits', scatter=False)

Basically, HPC-OFC theta coherence decreases while HPC-OFC beta coherence increases when encoding positive RPEs into memory.

## Second, let's compute this for a sliding window and perform regression

In [ ]:

%%capture
# from IPython.display import display

pairs = (['HPC', 'OFC'],)
# pairs = ['AMY', 'HPC'], 
# pairs = (['HPC', 'OFC'], )

win_len = 200  # ms 
slide_len = 50 # ms 
buf_ms = 1000

for pair in pairs:
    source_region = pair[0]
    target_region = pair[1]
    
    # initialize the storage list 
    conn_group_data = []

    # iterate through subjects
    for subj in tqdm(subj_ids, desc=f'{source_region} to {target_region} connectivity using {metric}'):
        combined_df.loc[combined_df.participant==subj, 
                'zrpe'] = (combined_df.loc[combined_df.participant==subj, 
                                                                        'rpe'] - np.nanmean(combined_df.loc[combined_df.participant==subj, 
                                                                                                'rpe'])) / (2*np.nanstd(combined_df.loc[combined_df.participant==subj, 
                                                                                                                                    'rpe']))
        combined_df.loc[combined_df.participant==subj, 
                'zpm'] = (combined_df.loc[combined_df.participant==subj, 
                                                                        'DPRIME'] - np.nanmean(combined_df.loc[combined_df.participant==subj, 
                                                                                                'DPRIME'])) / (2*np.nanstd(combined_df.loc[combined_df.participant==subj, 
                                                                                                                                    'DPRIME']))            
        combined_df['phit'] = combined_df.apply(lambda x: 0.5 if x.hits==True else -0.5, axis=1)
        regressors = combined_df.loc[combined_df.participant==subj, ['participant', 'zrpe', 'zpm', 'phit', 'trials_gamble']]

        # display(f'Computing granger causality for {pair} in {subj_id}')
        for event in analysis_evs:
            loadpath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj}'
            epochs_reref = mne.read_epochs(f'{loadpath}/neural/{day}/{event}-epo.fif', preload=True) 

            # Crop the data to the relevant time window
            buf_s = buf_ms / 1000
            epochs_reref.crop(tmin=epochs_reref.tmin + buf_s,
                        tmax=epochs_reref.tmax - buf_s)
            
            # define slices for smoothing timeseries: 
            slices = np.lib.stride_tricks.sliding_window_view(np.arange(epochs_reref._data.shape[-1]), win_len)[::slide_len]
            midpoints = np.ceil(np.mean(slices, axis=1))
            ts = midpoints * (1000/epochs_reref.info['sfreq'])


            # Get electrode df 
            elec_df = pd.read_csv(f'{loadpath}/Day1_reref_elec_df')

            seed_target_df = oscillation_utils.make_seed_target_df(elec_df, epochs_reref, source_region, target_region)
            
            # For within-region connectivity, let's drop elements that are the same
            
            if len(seed_target_df) == 0:
                continue
                
            # for cond in conditions: 
            for hemi in ['l', 'r']:
                # first determine if ipsilateral connectivity is even possible; if not, move on. 
                # note: edit this if you want to compute contralateral connectivity 
                if hemi not in seed_target_df.index.tolist():
                    continue
                else:
                    seed_to_target = seed_target_indices(
                        seed_target_df['seed'][hemi],
                        seed_target_df['target'][hemi])

                npairs = len(seed_to_target[0])

                sig = np.zeros([epochs_reref._data.shape[0], npairs, slices.shape[0]])
                for stride in range(slices.shape[0]):

                    ts_start = (1000/epochs_reref.info['sfreq']) * (slices[stride][0]/1000)
                    ts_end = (1000/epochs_reref.info['sfreq']) * (slices[stride][-1]/1000)

                    pwise_epochs = epochs_reref.copy().crop(tmin=ts_start, tmax=ts_end)

                    pwise = oscillation_utils.amp_amp_coupling(pwise_epochs, 
                                                seed_to_target, 
                                                freqs0=[8, 13],
                                                freqs1=[8, 13])
                    sig[:, :, stride] = pwise
            

                for j in range(npairs):
                    for i in range(len(ts)):
                        ch_df = regressors.copy()
                        ch_df['ts'] = ts[i]
                        ch_df.rename(columns={'trials_gamble':'trial'}, inplace=True)
                        # ch_df.dropna()
                        ch_df.sort_values(by=['trial'], inplace=True)
                        trial_ix = ch_df.trial.values.astype(int) - 1
                        ch_df['conn'] = sig[trial_ix, j, i]
                        ch_df['pair_label'] = subj + '_' + epochs_reref.ch_names[seed_to_target[0][j]] + ':' + epochs_reref.ch_names[seed_to_target[1][j]]
                        
                        conn_group_data.append(ch_df)

    full_df = pd.concat(conn_group_data).dropna(subset=['zrpe'])
    full_df['band'] = 'alpha'
    full_df['source_region'] = source_region
    full_df['target_region'] = target_region


In [ ]:
sns.pointplot(data=full_df[full_df.zrpe>0], x='ts', y='conn', hue='phit')

In [ ]:
all_res = statistics_utils.time_resolved_mlm(full_df,
                    y='conn',
                    formula='conn ~ 1 + zrpe*phit',
                    lower_group='pair_label',
                    higher_group='participant',
                    trial_key='trial',
                    n_permutations=100)


In [ ]:
all_res.to_csv(f'{load_dir}/all_res_time_resolved_mlm_conn_{source_region}_{target_region}.csv', index=False)


In [ ]:
from statsmodels.stats.multitest import multipletests
all_res['mc_p'] = np.nan
all_res.loc[all_res.parameter=='zrpe:phit', 'mc_p'] = multipletests(all_res.loc[all_res.parameter=='zrpe:phit', 'z_p'], method='fdr_by')[1]

In [ ]:
for ts in full_df.ts.unique():

    test_model = smf.mixedlm(data=full_df[full_df.ts==ts],
            formula='conn ~ 1 + zrpe*phit',
            groups='pair_label').fit()

    if test_model.pvalues['zrpe:phit'] <= 0.05:
    sns.lmplot(data=full_df[full_df.ts==ts], x='zrpe', y='conn', hue='phit', scatter=False, ci=None)
    plt.title(f'ts={ts}')


Connectivity:

1. HPC-OFC is the only one with consistent coherence change (1000-1200 ms); increae with rpe for memory



In [ ]:
all_res.loc[all_res.parameter=='zrpe:phit']